# Xiangqi — terminal replay comparison

Same 20 training boards and 6 validation boards. Only replay sampling changes.

## 1. Load rules and boards

Loads the project source snapshot and checks the fixed board split.

In [ ]:
# @title Load rules and boards
import hashlib, importlib, json, sys
from pathlib import Path
SOURCE_FILES = {'rules.py': '"""Dependency-free Xiangqi rules and a reusable, turn-based experiment state.\n\nCoordinates: board[row][col], row 0 is Black\'s home rank; UCI a0 is Red\'s.\nResearch draw policy: threefold position repetition / 120 non-capture plies.\nThis is not the full competition perpetual-check/chase adjudicator.\n"""\nfrom collections import Counter\nimport copy\nimport re\n\nSTART_FEN = "rheakaehr/9/1c5c1/p1p1p1p1p/9/9/P1P1P1P1P/1C5C1/9/RHEAKAEHR w - - 0 1"\nPIECES = "KAEHRCPkaehrcp"\nPRESETS = [\n    {"id": "rook", "name": "Rook & pawn", "tag": "Starter · 6 pieces", "description": "Coordinate a rook and an advancing pawn.", "fen": "4k4/4a4/9/9/4p4/2P6/9/4R4/9/3K5 w - - 0 1"},\n    {"id": "cannon", "name": "Cannon coordination", "tag": "Explore · 8 pieces", "description": "Explore cannon screens and pawn coordination.", "fen": "3aka3/9/9/4p4/9/4P4/2C6/9/9/3AK4 w - - 0 1"},\n    {"id": "horse", "name": "Horse & pawn", "tag": "Explore · 8 pieces", "description": "Control horse paths and key squares.", "fen": "3ak4/9/4e4/9/4p4/2P6/5H3/9/9/3AK4 w - - 0 1"},\n    {"id": "start", "name": "Starting position", "tag": "Reference · 32 pieces", "description": "A full board for demonstrations and rule checks.", "fen": START_FEN},\n]\n\ndef color(piece):\n    return "red" if piece.isupper() else "black"\n\ndef other(side):\n    return "black" if side == "red" else "red"\n\ndef square(row, col):\n    return chr(97 + col) + str(9 - row)\n\ndef coords(name):\n    if not re.fullmatch(r"[a-i][0-9]", name):\n        raise ValueError("Coordinates must be between a0 and i9.")\n    return 9 - int(name[1]), ord(name[0]) - 97\n\ndef action_id(move):\n    r1, c1 = coords(move[:2]); r2, c2 = coords(move[2:])\n    return (r1 * 9 + c1) * 90 + r2 * 9 + c2\n\ndef action_move(action):\n    if type(action) is not int or not 0 <= action < 8100:\n        raise ValueError("Action IDs must be between 0 and 8099.")\n    src, dst = divmod(action, 90)\n    return square(*divmod(src, 9)) + square(*divmod(dst, 9))\n\ndef inside(r, c):\n    return 0 <= r < 10 and 0 <= c < 9\n\ndef palace(r, c, side):\n    return 3 <= c <= 5 and (7 <= r <= 9 if side == "red" else 0 <= r <= 2)\n\ndef pseudo(board, r, c, tr, tc):\n    piece = board[r][c]\n    if not piece or (r, c) == (tr, tc) or not inside(tr, tc):\n        return False\n    target = board[tr][tc]\n    side = color(piece)\n    if target and color(target) == side:\n        return False\n    dr, dc = tr-r, tc-c\n    ar, ac = abs(dr), abs(dc)\n    kind = piece.lower()\n    blockers = -1\n    if r == tr:\n        blockers = sum(bool(board[r][i]) for i in range(min(c, tc)+1, max(c, tc)))\n    elif c == tc:\n        blockers = sum(bool(board[i][c]) for i in range(min(r, tr)+1, max(r, tr)))\n    if kind == "r":\n        return blockers == 0\n    if kind == "c":\n        return blockers == (1 if target else 0)\n    if kind == "h":\n        if (ar, ac) == (2, 1):\n            return not board[r + dr//2][c]\n        if (ar, ac) == (1, 2):\n            return not board[r][c + dc//2]\n        return False\n    if kind == "e":\n        return ar == ac == 2 and (tr >= 5 if side == "red" else tr <= 4) and not board[r+dr//2][c+dc//2]\n    if kind == "a":\n        return ar == ac == 1 and palace(tr, tc, side)\n    if kind == "k":\n        return (ar+ac == 1 and palace(tr, tc, side)) or (target and target.lower() == "k" and c == tc and blockers == 0)\n    if kind == "p":\n        return (dr == (-1 if side == "red" else 1) and dc == 0) or (dr == 0 and ac == 1 and (r <= 4 if side == "red" else r >= 5))\n    return False\n\ndef in_check(board, side):\n    king = "K" if side == "red" else "k"\n    pos = next(((r,c) for r in range(10) for c in range(9) if board[r][c] == king), None)\n    if pos is None:\n        return True\n    return any(p and color(p) != side and pseudo(board,r,c,*pos) for r,row in enumerate(board) for c,p in enumerate(row))\n\ndef legal_moves(board, side):\n    moves = []\n    for r,row in enumerate(board):\n        for c,p in enumerate(row):\n            if not p or color(p) != side:\n                continue\n            for tr in range(10):\n                for tc in range(9):\n                    target = board[tr][tc]\n                    if target.lower() == "k" or not pseudo(board,r,c,tr,tc):\n                        continue\n                    board[r][c], board[tr][tc] = "", p\n                    safe = not in_check(board,side)\n                    board[r][c], board[tr][tc] = p, target\n                    if safe:\n                        moves.append(square(r,c)+square(tr,tc))\n    return moves\n\ndef parse_fen(fen):\n    if not isinstance(fen,str) or len(fen) > 250:\n        raise ValueError("Invalid FEN format.")\n    fields = fen.strip().split()\n    if len(fields) not in (2,6) or fields[1] not in ("w","r","b"):\n        raise ValueError("FEN requires a board and side w/r/b, optionally followed by - - halfmove fullmove.")\n    rows = fields[0].replace("n","h").replace("N","H").replace("b","e").replace("B","E").split("/")\n    if len(rows) != 10:\n        raise ValueError("FEN must contain 10 ranks.")\n    board = []\n    for row in rows:\n        result = []\n        for ch in row:\n            if ch in "123456789": result.extend([""] * int(ch))\n            elif ch in PIECES: result.append(ch)\n            else: raise ValueError("FEN contains an unknown piece.")\n        if len(result) != 9: raise ValueError("Each FEN rank must contain 9 squares.")\n        board.append(result)\n    side = "black" if fields[1] == "b" else "red"\n    half, full = 0, 1\n    if len(fields) == 6:\n        if fields[2:4] != ["-","-"]: raise ValueError("FEN fields 3 and 4 must be - -.")\n        try: half, full = int(fields[4]), int(fields[5])\n        except ValueError: raise ValueError("FEN move counters must be integers.")\n        if not 0 <= half <= 100000 or not 1 <= full <= 100000: raise ValueError("Move counters are out of range.")\n    counts = Counter(p for row in board for p in row if p)\n    for p,limit in zip("KAEHRCP",[1,2,2,2,2,2,5]):\n        if counts[p] > limit or counts[p.lower()] > limit: raise ValueError("Piece counts exceed standard Xiangqi limits.")\n    if counts["K"] != 1 or counts["k"] != 1: raise ValueError("Each side must have exactly one king.")\n    for r,row in enumerate(board):\n        for c,p in enumerate(row):\n            if not p: continue\n            s = color(p)\n            if p.lower() in "ka" and not palace(r,c,s): raise ValueError("Kings and advisors must be inside their own palace.")\n            if p.lower() == "a" and (r,c) not in ([(9,3),(9,5),(8,4),(7,3),(7,5)] if s == "red" else [(0,3),(0,5),(1,4),(2,3),(2,5)]): raise ValueError("An advisor is on an invalid square.")\n            if p.lower() == "e" and (r,c) not in ([(9,2),(9,6),(7,0),(7,4),(7,8),(5,2),(5,6)] if s == "red" else [(0,2),(0,6),(2,0),(2,4),(2,8),(4,2),(4,6)]): raise ValueError("An elephant is on an invalid square.")\n            if p.lower() == "p" and ((s == "red" and (r > 6 or (r >= 5 and c%2))) or (s == "black" and (r < 3 or (r <= 4 and c%2)))): raise ValueError("A pawn is on an unreachable square.")\n    if in_check(board, other(side)):\n        raise ValueError("The non-moving side is in check or the kings face each other. Adjust the position or side to move.")\n    return board, side, half, full\n\ndef make_fen(board, side, half=0, full=1):\n    rows=[]\n    for row in board:\n        out=""; empty=0\n        for p in row:\n            if not p: empty+=1\n            else:\n                if empty: out+=str(empty); empty=0\n                out+=p\n        if empty: out+=str(empty)\n        rows.append(out)\n    return "/".join(rows)+f" {\'w\' if side == \'red\' else \'b\'} - - {half} {full}"\n\nclass Game:\n    def __init__(self, fen=START_FEN, max_plies=160):\n        self.board, self.side, self.halfmove, self.fullmove = parse_fen(fen)\n        self.initial_fen = self.fen\n        self.max_plies = max_plies\n        self.history=[]\n        self.repetitions=Counter({self.key:1})\n        self._legal=None\n\n    @property\n    def fen(self): return make_fen(self.board,self.side,self.halfmove,self.fullmove)\n    @property\n    def key(self): return " ".join(self.fen.split()[:2])\n    @property\n    def legal(self):\n        if self._legal is None: self._legal=legal_moves(self.board,self.side)\n        return self._legal\n\n    def outcome(self):\n        if not self.legal:\n            return {"winner":other(self.side), "reason":"checkmate" if in_check(self.board,self.side) else "stalemate", "terminated":True, "truncated":False}\n        if self.repetitions[self.key] >= 3: reason="repetition"\n        elif self.halfmove >= 120: reason="no_capture"\n        elif len(self.history) >= self.max_plies:\n            return {"winner":None,"reason":"ply_limit","terminated":False,"truncated":True}\n        else: return None\n        return {"winner":None,"reason":reason,"terminated":True,"truncated":False}\n\n    def step(self, move, actor="human", analysis=None):\n        if self.outcome(): raise ValueError("The episode has ended. Reset or load another position.")\n        if move not in self.legal: raise ValueError("Illegal move: you cannot leave your king in check or expose facing kings.")\n        r,c=coords(move[:2]); tr,tc=coords(move[2:])\n        entry={"move":move,"side":self.side,"piece":self.board[r][c],"captured":self.board[tr][tc],"fen_before":self.fen,"actor":actor,"analysis":analysis or {}}\n        self.board[tr][tc], self.board[r][c] = self.board[r][c], ""\n        self.halfmove=0 if entry["captured"] else self.halfmove+1\n        if self.side == "black": self.fullmove+=1\n        self.side=other(self.side); self._legal=None\n        self.repetitions[self.key]+=1\n        entry["fen_after"]=self.fen\n        self.history.append(entry)\n        return self.outcome()\n\n    def undo(self):\n        if not self.history: return\n        self.repetitions[self.key]-=1\n        entry=self.history.pop()\n        self.board,self.side,self.halfmove,self.fullmove=parse_fen(entry["fen_before"])\n        self._legal=None\n\n    def observation(self):\n        planes=[[[int(p == kind) for p in row] for row in self.board] for kind in PIECES]\n        return {"fen":self.fen,"board_planes":planes,"plane_order":PIECES,"side_to_move":self.side,"legal_moves":list(self.legal),"legal_action_ids":[action_id(m) for m in self.legal],"halfmove_clock":self.halfmove,"ply":len(self.history),"max_plies":self.max_plies,"repetition_counts":dict(self.repetitions)}\n\n    def snapshot(self):\n        return {"fen":self.fen,"initial_fen":self.initial_fen,"board":copy.deepcopy(self.board),"side":self.side,"legal_moves":list(self.legal),"in_check":in_check(self.board,self.side),"history":copy.deepcopy(self.history),"outcome":self.outcome(),"halfmove_clock":self.halfmove,"max_plies":self.max_plies,"piece_count":sum(bool(p) for row in self.board for p in row)}\n\n    def export(self):\n        return {"schema":"xiangqi-lab.episode.v1","rules":"xiangqi-research-v1","draw_policy":"threefold-position-or-120-noncapture-plies","initial_fen":self.initial_fen,"max_plies":self.max_plies,"moves":copy.deepcopy(self.history),"final_fen":self.fen,"outcome":self.outcome()}\n', 'training.py': '"""Turn-based two-player environment; no ML framework required.\n\nreset -> (observation, info); step -> (observation, reward, terminated, truncated, info)\nReward is from the ACTING side\'s perspective, not always Red\'s.\nThis API resembles Gymnasium but is not a gymnasium.Env subclass.\n"""\nfrom rules import Game, START_FEN, action_move\n\nclass XiangqiEnv:\n    def __init__(self, max_plies=160):\n        self.max_plies=max_plies\n        self.game=None\n\n    def reset(self, *, fen=START_FEN, seed=None):\n        self.game=Game(fen,self.max_plies)\n        return self.game.observation(), {"outcome":self.game.outcome(),"seed":seed}\n\n    def step(self, action):\n        if self.game is None: raise RuntimeError("Call reset before step.")\n        move=action_move(action) if type(action) is int else action\n        side=self.game.side\n        result=self.game.step(move,actor="training")\n        reward=0.0\n        if result and result["winner"]:\n            reward=1.0 if result["winner"] == side else -1.0\n        return self.game.observation(), reward, bool(result and result["terminated"]), bool(result and result["truncated"]), {"acting_side":side,"outcome":result,"rewards":{"red": (1 if result and result["winner"] == "red" else -1 if result and result["winner"] == "black" else 0),"black": (1 if result and result["winner"] == "black" else -1 if result and result["winner"] == "red" else 0)}}\n', 'notebook_tools.py': '"""Small, standard-library helpers for the environment notebook (no trainer)."""\nfrom html import escape\n\nfrom rules import Game, PRESETS\n\n\nENDGAMES = {p["id"]: dict(p) for p in PRESETS if p["id"] in ("rook", "cannon", "horse")}\nPIECE_LABELS = dict(zip("KAEHRCPkaehrcp", "帥仕相馬車炮兵將士象馬車砲卒"))\n\n\ndef action_mask(env):\n    """An 8100-entry mask; an ended episode has no available actions."""\n    if env.game is None:\n        raise RuntimeError("Call reset before asking for legal actions.")\n    mask = [False] * 8100\n    if env.game.outcome() is None:\n        for index in env.game.observation()["legal_action_ids"]:\n            mask[index] = True\n    return mask\n\n\ndef replay_episode(episode):\n    """Replay the website\'s JSON format, validating every move and final FEN."""\n    if not isinstance(episode, dict) or episode.get("schema") != "xiangqi-lab.episode.v1":\n        raise ValueError("Unsupported episode format.")\n    moves = episode.get("moves", [])\n    limit = episode.get("max_plies", 160)\n    if not isinstance(moves, list) or len(moves) > 1000:\n        raise ValueError("Invalid move list.")\n    if type(limit) is not int or not 2 <= limit <= 1000:\n        raise ValueError("The ply limit must be between 2 and 1000.")\n    game = Game(episode.get("initial_fen", ""), limit)\n    for entry in moves:\n        if not isinstance(entry, dict):\n            raise ValueError("Invalid move record.")\n        game.step(entry.get("move", ""), actor=str(entry.get("actor", "import"))[:100])\n    if episode.get("final_fen") and game.fen != episode["final_fen"]:\n        raise ValueError("The move history does not match the final FEN.")\n    return game\n\n\ndef board_svg(game, title="Xiangqi Lab"):\n    """Browser-rendered SVG: Red at the bottom, traditional piece symbols."""\n    title = escape(str(title))\n    parts = [\n        \'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 620 744" \'\n        \'width="620" role="img" aria-label="Xiangqi board">\',\n        f\'<title>{title}</title>\',\n        \'<rect width="620" height="744" rx="18" fill="#f6f1e8"/>\',\n        \'<g font-family="Arial, sans-serif" fill="#29423e">\',\n        f\'<text x="40" y="38" font-size="22" font-weight="bold">{title}</text>\',\n        f\'<text x="40" y="65" font-size="15">Turn: {game.side.title()} · Move count: {len(game.history)}</text>\',\n        \'</g>\',\n        \'<rect x="32" y="85" width="556" height="612" rx="10" fill="#efe1c7"/>\',\n        \'<g stroke="#a68b61" stroke-width="1.4">\',\n    ]\n\n    def line(x1, y1, x2, y2):\n        parts.append(f\'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}"/>\')\n\n    for row in range(10):\n        line(70, 120 + row * 60, 550, 120 + row * 60)\n    for col in range(9):\n        x = 70 + col * 60\n        if col in (0, 8):\n            line(x, 120, x, 660)\n        else:\n            line(x, 120, x, 360)\n            line(x, 420, x, 660)\n    for y in (120, 540):\n        line(250, y, 370, y + 120)\n        line(370, y, 250, y + 120)\n    parts += [\'</g>\', \'<g font-family="Arial, sans-serif" text-anchor="middle" fill="#806a48">\',\n              \'<text x="310" y="397" font-size="18" letter-spacing="6">RIVER</text>\']\n    for col in range(9):\n        parts.append(f\'<text x="{70 + col * 60}" y="688" font-size="13">{chr(97 + col)}</text>\')\n    for row in range(10):\n        parts.append(f\'<text x="20" y="{125 + row * 60}" font-size="13">{9 - row}</text>\')\n    parts.append(\'</g>\')\n    for row, cells in enumerate(game.board):\n        for col, piece in enumerate(cells):\n            if not piece:\n                continue\n            x, y = 70 + col * 60, 120 + row * 60\n            ink = "#ab3e2f" if piece.isupper() else "#29423e"\n            parts.extend([\n                f\'<g><title>{piece} on {chr(97 + col)}{9 - row}</title>\',\n                f\'<circle cx="{x}" cy="{y}" r="25" fill="#fff6dc" stroke="{ink}" stroke-width="1.4"/>\',\n                f\'<circle cx="{x}" cy="{y}" r="21.5" fill="none" stroke="#d7c49e"/>\',\n                f\'<text x="{x}" y="{y + 11}" text-anchor="middle" font-size="32" \'\n                f\'font-family="Noto Serif CJK SC, Microsoft YaHei, PingFang SC, serif" fill="{ink}">{PIECE_LABELS[piece]}</text></g>\',\n            ])\n    result = game.outcome()\n    footer = "Game in progress" if result is None else f"End: {result[\'reason\']} · Winner: {result[\'winner\'] or \'none\'}"\n    parts.append(f\'<text x="40" y="726" font-size="15" font-family="Arial, sans-serif" fill="#29423e">{escape(footer)}</text></svg>\')\n    return "\\n".join(parts)\n', 'dqn_model.py': '"""DQN network and stateless website policy; PyTorch is an optional dependency."""\nfrom functools import lru_cache\n\nimport torch\nfrom torch import nn\n\nMODEL_SCHEMA = "xiangqi-lab.dqn.v1"\nPIECE_ORDER = "KAEHRCPkaehrcp"\nACTION_COUNT = 8100\n\n\n@lru_cache(maxsize=32768)\ndef position_tokens(key):\n    """Encode a position key without changing orientation or action numbering."""\n    board, side = key.split()[:2]\n    tokens = []\n    for char in board:\n        if char == "/":\n            continue\n        if char in "123456789":\n            tokens.extend([0] * int(char))\n        else:\n            tokens.append(PIECE_ORDER.index(char) + 1)\n    if len(tokens) != 90 or side not in ("w", "b", "r"):\n        raise ValueError("Invalid position key.")\n    return bytes(tokens), int(side == "b")\n\n\ndef encode_state(observation):\n    """Keep every repetition entry, not only the current position\'s count.\n\n    Values are standard Python types so replay can be saved safely with torch.\n    The external episode time limit is not an input to the continuing-game value.\n    """\n    key = " ".join(observation["fen"].split()[:2])\n    board, side = position_tokens(key)\n    history = tuple((*position_tokens(k), int(count))\n                    for k, count in sorted(observation["repetition_counts"].items()) if count > 0)\n    if not history:\n        raise ValueError("The observation must include repetition counts.")\n    return board, side, float(observation["halfmove_clock"]) / 120.0, history\n\n\ndef batch_states(states, device):\n    boards, metadata, history_boards, history_meta, owners = [], [], [], [], []\n    for owner, (board, side, clock, history) in enumerate(states):\n        boards.append(list(board))\n        metadata.append([side, clock, len(history) / 160.0])\n        for old_board, old_side, count in history:\n            history_boards.append(list(old_board))\n            history_meta.append([old_side, count / 3.0])\n            owners.append(owner)\n    return (\n        torch.tensor(boards, dtype=torch.long, device=device),\n        torch.tensor(metadata, dtype=torch.float32, device=device),\n        torch.tensor(history_boards, dtype=torch.long, device=device),\n        torch.tensor(history_meta, dtype=torch.float32, device=device),\n        torch.tensor(owners, dtype=torch.long, device=device),\n    )\n\n\nclass QNetwork(nn.Module):\n    """Score all action IDs, using the current board and the full repetition map."""\n    def __init__(self, hidden=64, embedding=4, history_hidden=16):\n        super().__init__()\n        self.config = {"hidden": hidden, "embedding": embedding, "history_hidden": history_hidden}\n        self.pieces = nn.Embedding(15, embedding, padding_idx=0)\n        self.current = nn.Sequential(nn.Linear(90 * embedding + 3, hidden), nn.ReLU())\n        self.history = nn.Sequential(nn.Linear(90 * embedding + 2, history_hidden), nn.ReLU())\n        self.head = nn.Sequential(nn.Linear(hidden + history_hidden, hidden), nn.ReLU(),\n                                  nn.Linear(hidden, ACTION_COUNT))\n\n    def forward(self, batch):\n        boards, metadata, old_boards, old_meta, owners = batch\n        current = self.current(torch.cat((self.pieces(boards).flatten(1), metadata), dim=1))\n        old = self.history(torch.cat((self.pieces(old_boards).flatten(1), old_meta), dim=1))\n        # Each complete repetition map is pooled as an unordered set.\n        pooled = old.new_zeros((len(boards), old.shape[1])).index_add(0, owners, old)\n        count = torch.bincount(owners, minlength=len(boards)).clamp_min(1).sqrt().unsqueeze(1)\n        return self.head(torch.cat((current, pooled / count), dim=1))\n\n\n@torch.no_grad()\ndef greedy_action(model, state, legal_ids, device):\n    if not legal_ids:\n        raise ValueError("No legal action is available.")\n    q = model(batch_states([state], device))[0]\n    legal = torch.tensor(legal_ids, dtype=torch.long, device=device)\n    return int(legal[q[legal].argmax()].item())\n\n\nclass Policy:\n    """Load an exported checkpoint once and serve the website\'s Python protocol."""\n    def __init__(self, checkpoint=None, device="cpu"):\n        if not checkpoint:\n            raise ValueError("A trained DQN checkpoint is required.")\n        torch.set_num_threads(2)\n        self.device = torch.device(device)\n        saved = torch.load(checkpoint, map_location="cpu", weights_only=True)\n        if saved.get("schema") != MODEL_SCHEMA:\n            raise ValueError("Unsupported DQN checkpoint format.")\n        self.model = QNetwork(**saved["network_config"]).to(self.device)\n        self.model.load_state_dict(saved["model"])\n        self.model.eval()\n\n    def select_move(self, observation, time_budget_ms=800):\n        return {"action_id": greedy_action(self.model, encode_state(observation),\n                                           observation["legal_action_ids"], self.device)}\n', 'dqn_training.py': '"""Fixed-opponent DQN and Double DQN with validation and early stopping."""\nfrom collections import deque\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nimport csv\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport zipfile\n\nimport torch\nfrom torch import nn\n\nfrom dqn_model import MODEL_SCHEMA, QNetwork, batch_states, encode_state, greedy_action\nfrom rules import Game, PRESETS, action_id, action_move\n\nENDGAMES = {p["id"]: p for p in PRESETS if p["id"] in ("rook", "cannon", "horse")}\n\n\nclass FixedOpponentEnv:\n    """One learner decision and (if needed) a fixed-policy opponent reply."""\n    def __init__(self, max_plies=160, learner="red", opponent=None):\n        if learner not in ("red", "black") or type(max_plies) is not int or not 4 <= max_plies <= 1000:\n            raise ValueError("Use red/black and a ply limit from 4 to 1000.")\n        self.max_plies, self.learner = max_plies, learner\n        self.opponent = opponent\n        self.game = None\n\n    def _reply(self):\n        move = (self.opponent(self.game.observation(), self.rng) if self.opponent\n                else self.rng.choice(self.game.legal))\n        self.game.step(move, actor="fixed-opponent")\n\n    def reset(self, fen, seed=0):\n        self.rng = random.Random(seed)\n        self.game = Game(fen, self.max_plies)\n        opening = int(self.game.side != self.learner)\n        # Align the limit to the learner\'s next turn, so a time-limit transition\n        # can bootstrap from a learner state. Never exceed the requested cap.\n        self.game.max_plies = self.max_plies - ((self.max_plies - opening) % 2)\n        if opening and self.game.outcome() is None:\n            self._reply()\n        return self.game.observation(), {"outcome": self.game.outcome()}\n\n    def step(self, action):\n        if self.game is None or self.game.outcome() is not None:\n            raise ValueError("Reset before moving in a new episode.")\n        if self.game.side != self.learner:\n            raise RuntimeError("A training step must start on the learner\'s turn.")\n        before = len(self.game.history)\n        move = action_move(action) if type(action) is int else action\n        self.game.step(move, actor="dqn")\n        if self.game.outcome() is None:\n            self._reply()\n        outcome = self.game.outcome()\n        winner = outcome["winner"] if outcome else None\n        reward = (1.0 if winner == self.learner else -1.0) if winner else 0.0\n        return (self.game.observation(), reward,\n                bool(outcome and outcome["terminated"]), bool(outcome and outcome["truncated"]),\n                {"outcome": outcome, "learner": self.learner,\n                 "environment_plies": len(self.game.history) - before})\n\n\n@dataclass\nclass TrainConfig:\n    preset: str = "rook"\n    learner: str = "red"\n    min_episodes: int = 1000\n    max_episodes: int = 3000\n    max_plies: int = 160\n    eval_every: int = 100\n    eval_games: int = 50\n    patience: int = 5\n    min_delta: float = 0.02\n    seed: int = 42\n    learning_rate: float = 0.0003\n    gamma: float = 0.99\n    replay_size: int = 10000\n    batch_size: int = 64\n    warmup_steps: int = 512\n    train_every: int = 4\n    target_every: int = 250\n    epsilon_start: float = 1.0\n    epsilon_end: float = 0.1\n    epsilon_decay_steps: int = 20000\n    device: str = "auto"\n    algorithm: str = "dqn"\n    replay_sampling: str = "uniform"\n    terminal_mix: float = 0.5\n\n    def validate(self):\n        if self.algorithm not in ("dqn", "double_dqn"):\n            raise ValueError("Choose dqn or double_dqn.")\n        if self.replay_sampling not in ("uniform", "terminal_mixture"):\n            raise ValueError("Choose uniform or terminal_mixture replay sampling.")\n        if not math.isfinite(self.terminal_mix) or not 0 <= self.terminal_mix < 1:\n            raise ValueError("terminal_mix must be finite and in [0, 1).")\n        if self.preset not in ENDGAMES or self.learner not in ("red", "black"):\n            raise ValueError("Unknown preset or learner colour.")\n        for name in ("min_episodes", "max_episodes", "max_plies", "eval_every", "eval_games",\n                     "patience", "replay_size", "batch_size", "warmup_steps", "train_every",\n                     "target_every", "epsilon_decay_steps"):\n            if type(getattr(self, name)) is not int or getattr(self, name) < 1:\n                raise ValueError(name + " must be a positive integer.")\n        if self.max_episodes < self.min_episodes:\n            raise ValueError("MAX_EPISODES must be at least MIN_EPISODES.")\n        if not 4 <= self.max_plies <= 1000 or self.replay_size < self.batch_size:\n            raise ValueError("Invalid ply limit or replay size.")\n        if not 0 <= self.epsilon_end <= self.epsilon_start <= 1 or not 0 <= self.gamma <= 1:\n            raise ValueError("Invalid epsilon or discount factor.")\n        if not math.isfinite(self.learning_rate) or self.learning_rate <= 0:\n            raise ValueError("Learning rate must be positive and finite.")\n        if not math.isfinite(self.min_delta) or self.min_delta < 0:\n            raise ValueError("Minimum improvement must be finite and non-negative.")\n\n\n@dataclass\nclass EarlyStopping:\n    min_episodes: int\n    patience: int\n    min_delta: float\n    best_score: float = -math.inf\n    reference_score: float = -math.inf\n    bad_evals: int = 0\n    monitoring_started: bool = False\n\n    def update(self, episode, score):\n        if not math.isfinite(score):\n            raise ValueError("Validation score must be finite.")\n        improved = score > self.best_score\n        if improved:\n            self.best_score = score\n        if episode < self.min_episodes:\n            self.bad_evals = 0\n        elif not self.monitoring_started:\n            self.monitoring_started = True\n            self.reference_score = self.best_score\n            self.bad_evals = 0\n        elif score > self.reference_score + self.min_delta:\n            self.reference_score = score\n            self.bad_evals = 0\n        else:\n            self.bad_evals += 1\n        return improved, self.monitoring_started and self.bad_evals >= self.patience\n\n\n@torch.no_grad()\ndef legal_next_values(target, next_states, legal_ids, terminated, device, online=None):\n    """Double DQN selects with online and evaluates with target, using legal moves."""\n    values = torch.zeros(len(next_states), device=device)\n    live = [i for i, terminal in enumerate(terminated) if not terminal]\n    if live:\n        batch = batch_states([next_states[i] for i in live], device)\n        scores = target(batch)\n        selection = online(batch) if online is not None else scores\n        for row, index in enumerate(live):\n            if not legal_ids[index]:\n                raise ValueError("A nonterminal transition needs legal next actions.")\n            legal = torch.tensor(legal_ids[index], dtype=torch.long, device=device)\n            chosen = legal[selection[row, legal].argmax()]\n            values[index] = scores[row, chosen]\n    return values\n\n\ndef transition_group(transition):\n    """Time-limit truncations are live transitions, not terminal draws."""\n    if not transition[4]:\n        return "live"\n    return "win" if transition[2] > 0 else "loss" if transition[2] < 0 else "draw"\n\n\ndef sample_replay(replay, rng, config):\n    """Mix uniform replay with terminal groups; correct by exact sampling odds."""\n    items = list(replay)\n    groups = {name: [] for name in ("win", "loss", "draw")}\n    for index, item in enumerate(items):\n        group = transition_group(item)\n        if group != "live":\n            groups[group].append(index)\n    active = [indices for indices in groups.values() if indices]\n    alpha = config.terminal_mix if config.replay_sampling == "terminal_mixture" and active else 0.0\n    counts = {name: len(indices) for name, indices in groups.items()}\n    counts["live"] = len(items) - sum(counts.values())\n    if not alpha:\n        # Keep the original uniform path and RNG sequence exactly unchanged.\n        batch = rng.sample(items, config.batch_size)\n        weights = [1.0] * len(batch)\n    else:\n        batch, weights = [], []\n        for _ in range(config.batch_size):\n            index = rng.choice(rng.choice(active)) if rng.random() < alpha else rng.randrange(len(items))\n            item = items[index]\n            group = transition_group(item)\n            probability = (1 - alpha) / len(items)\n            if group != "live":\n                probability += alpha / (len(active) * counts[group])\n            batch.append(item)\n            weights.append(1.0 / (len(items) * probability))\n    selected = {name: sum(transition_group(item) == name for item in batch) for name in counts}\n    return batch, weights, {"pool": counts, "selected": selected, "mixture_active": bool(alpha)}\n\n\ndef optimize(model, target, optimizer, replay, rng, config, device, sampling_stats=None):\n    batch, weights, sampled = sample_replay(replay, rng, config)\n    states, actions, rewards, next_states, terminals, next_ids = zip(*batch)\n    predicted = model(batch_states(states, device)).gather(\n        1, torch.tensor(actions, device=device).unsqueeze(1)).squeeze(1)\n    expected = torch.tensor(rewards, device=device) + config.gamma * legal_next_values(\n        target, next_states, next_ids, terminals, device,\n        online=model if config.algorithm == "double_dqn" else None)\n    if sampled["mixture_active"]:\n        loss = (nn.functional.smooth_l1_loss(predicted, expected, reduction="none") *\n                torch.tensor(weights, dtype=predicted.dtype, device=device)).mean()\n    else:\n        loss = nn.functional.smooth_l1_loss(predicted, expected)\n    if not torch.isfinite(loss):\n        raise FloatingPointError("Non-finite DQN loss; the run was stopped.")\n    optimizer.zero_grad(set_to_none=True)\n    loss.backward()\n    nn.utils.clip_grad_norm_(model.parameters(), 10.0)\n    optimizer.step()\n    if sampling_stats is not None:\n        sampling_stats["batches"] = sampling_stats.get("batches", 0) + 1\n        sampling_stats["mixture_batches"] = sampling_stats.get("mixture_batches", 0) + int(sampled["mixture_active"])\n        for name, count in sampled["selected"].items():\n            key = "sampled_" + name\n            sampling_stats[key] = sampling_stats.get(key, 0) + count\n        sampling_stats["last_pool"] = sampled["pool"]\n        sampling_stats["last_weight_min"] = min(weights)\n        sampling_stats["last_weight_max"] = max(weights)\n    return float(loss.item())\n\n\n@torch.no_grad()\ndef evaluate(model, config, device, split="validation", games=None, positions=None):\n    """Fixed seed bank, fresh envs, no exploration or training-state mutation."""\n    count = games or config.eval_games\n    if positions is not None and (not positions or count % len(positions)):\n        raise ValueError("Validation games must cover every board equally.")\n    stats = {"wins": 0, "losses": 0, "draws": 0, "truncations": 0, "environment_plies": 0}\n    per_board = {p["id"]: {key: 0 for key in ("wins", "losses", "draws", "truncations")} for p in positions or []}\n    previous_mode = model.training if model is not None else False\n    if model is not None:\n        model.eval()\n    try:\n        for index in range(count):\n            env = FixedOpponentEnv(config.max_plies, config.learner)\n            position = positions[index % len(positions)] if positions is not None else ENDGAMES[config.preset]\n            seed = (f"{split}:{position[\'id\']}:opponent:{config.seed}:{index // len(positions)}"\n                    if positions is not None else f"{split}:opponent:{config.seed}:{index}")\n            obs, info = env.reset(position["fen"], seed)\n            policy_rng = random.Random(f"{split}:random-policy:{config.seed}:{index}")\n            while info["outcome"] is None:\n                action = (policy_rng.choice(obs["legal_action_ids"]) if model is None\n                          else greedy_action(model, encode_state(obs), obs["legal_action_ids"], device))\n                obs, reward, terminated, truncated, info = env.step(action)\n            result = info["outcome"]\n            key = ("truncations" if result["truncated"] else "draws" if result["winner"] is None\n                   else "wins" if result["winner"] == config.learner else "losses")\n            stats[key] += 1\n            if positions is not None:\n                per_board[position["id"]][key] += 1\n            stats["environment_plies"] += len(env.game.history)\n    finally:\n        if model is not None:\n            model.train(previous_mode)\n    details = {"per_board": per_board} if positions is not None else {}\n    return {**stats, **details, "games": count, "split": split,\n            "score": (stats["wins"] - stats["losses"]) / count,\n            "win_rate": stats["wins"] / count}\n\n\ndef _write_json(path, value):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef _save_torch(path, value):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _append_csv(path, row):\n    exists = path.is_file()\n    with path.open("a", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(row))\n        if not exists:\n            writer.writeheader()\n        writer.writerow(row)\n\n\ndef train(config, run_dir, resume=False, source_hashes=None, board_bank=None, on_checkpoint=None):\n    """Run explicitly. No training starts when this module is imported."""\n    config.validate()\n    if board_bank is not None and config.eval_games % len(board_bank.validation):\n        raise ValueError("Validation games must cover every board equally.")\n    validation_positions = board_bank.validation if board_bank is not None else None\n    run_dir = Path(run_dir).resolve()\n    if resume:\n        if not (run_dir / "last.pt").is_file():\n            raise FileNotFoundError("No last.pt checkpoint in this run folder.")\n    elif run_dir.exists() and any(run_dir.iterdir()):\n        raise ValueError("Use a new run folder, or explicitly resume an existing run.")\n    run_dir.mkdir(parents=True, exist_ok=True)\n    torch.set_num_threads(2)\n    device = torch.device("cuda" if config.device == "auto" and torch.cuda.is_available()\n                          else "cpu" if config.device == "auto" else config.device)\n    torch.manual_seed(config.seed)\n    rng = random.Random(config.seed)\n    replay_rng = random.Random(config.seed + 1)\n    model, target = QNetwork().to(device), QNetwork().to(device)\n    target.load_state_dict(model.state_dict())\n    target.eval()\n    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)\n    replay = deque(maxlen=config.replay_size)\n    stopper = EarlyStopping(config.min_episodes, config.patience, config.min_delta)\n    steps = updates = total_plies = completed = 0\n    best_episode = None\n    last_loss = None\n    last_validation = None\n    sampling_stats = {}\n    if resume:\n        saved = torch.load(run_dir / "last.pt", map_location="cpu", weights_only=True)\n        if saved.get("schema") != MODEL_SCHEMA or saved.get("kind") != "training":\n            raise ValueError("Unsupported training checkpoint.")\n        for key, value in asdict(config).items():\n            defaults = {"algorithm": "dqn", "replay_sampling": "uniform", "terminal_mix": 0.5}\n            previous = saved["config"].get(key, defaults.get(key))\n            if key not in ("max_episodes", "device") and previous != value:\n                raise ValueError("Resume configuration differs: " + key)\n        if saved.get("source_hashes", {}) != (source_hashes or {}):\n            raise ValueError("Resume requires the same source snapshot.")\n        if saved.get("board_bank_sha256") != (board_bank.sha256 if board_bank is not None else None):\n            raise ValueError("Resume requires the same training and validation board bank.")\n        model.load_state_dict(saved["model"])\n        target.load_state_dict(saved["target"])\n        optimizer.load_state_dict(saved["optimizer"])\n        for state in optimizer.state.values():\n            for key, value in state.items():\n                if isinstance(value, torch.Tensor):\n                    state[key] = value.to(device)\n        replay.extend(saved["replay"])\n        rng.setstate(saved["rng"])\n        replay_rng.setstate(saved["replay_rng"])\n        torch.set_rng_state(saved["torch_rng"])\n        if device.type == "cuda" and saved.get("cuda_rng"):\n            torch.cuda.set_rng_state_all(saved["cuda_rng"])\n        stopper = EarlyStopping(**saved["early_stopping"])\n        completed, steps, updates, total_plies = (saved[k] for k in ("episode", "steps", "updates", "total_plies"))\n        best_episode, last_loss, last_validation = saved["best_episode"], saved["last_loss"], saved["validation"]\n        sampling_stats = saved.get("sampling_stats", {})\n        if completed >= config.max_episodes:\n            raise ValueError("The saved run already reached MAX_EPISODES. Increase it to resume.")\n        if not (run_dir / "best.pt").is_file():\n            raise FileNotFoundError("Resume needs the original best.pt beside last.pt.")\n        # CSV rows written after the last completed checkpoint may be from an\n        # interrupted episode range; remove those rows before replaying that range.\n        for name in ("episodes.csv", "validation.csv"):\n            path = run_dir / name\n            if path.exists():\n                with path.open(newline="", encoding="utf-8") as stream:\n                    reader = csv.DictReader(stream)\n                    fields, rows = reader.fieldnames, list(reader)\n                with path.open("w", newline="", encoding="utf-8") as stream:\n                    writer = csv.DictWriter(stream, fieldnames=fields)\n                    writer.writeheader()\n                    writer.writerows(row for row in rows if int(row["episode"]) <= completed)\n    else:\n        print("Checking the random-policy baseline...", flush=True)\n        _write_json(run_dir / "baseline.json", evaluate(None, config, device, positions=validation_positions))\n\n    if board_bank is not None:\n        _write_json(run_dir / "board_bank.json", board_bank.data)\n\n    _write_json(run_dir / "config.json", {**asdict(config), "actual_device": str(device),\n                "source_hashes": source_hashes or {}, "torch_version": str(torch.__version__),\n                "python_version": sys.version.split()[0], "opponent": "uniform-legal-random-v1",\n                "board_bank_sha256": board_bank.sha256 if board_bank is not None else None,\n                "validation_scope": "separate validation starting boards; fixed seeds; no test-board selection"\n                if board_bank is not None else "same preset; separate fixed random seeds; no unseen-board claim"})\n    started = time.monotonic()\n    stop_reason = "max_episodes"\n\n    def checkpoint(kind, validation):\n        payload = {"schema": MODEL_SCHEMA, "kind": kind, "network_config": model.config,\n                   "model": model.state_dict(), "config": asdict(config), "episode": completed,\n                   "steps": steps, "updates": updates, "total_plies": total_plies,\n                   "validation": validation, "source_hashes": source_hashes or {}}\n        if board_bank is not None:\n            payload["board_bank_sha256"] = board_bank.sha256\n        if kind == "training":\n            payload.update(target=target.state_dict(), optimizer=optimizer.state_dict(), replay=list(replay),\n                           rng=rng.getstate(), replay_rng=replay_rng.getstate(), torch_rng=torch.get_rng_state(),\n                           cuda_rng=torch.cuda.get_rng_state_all() if device.type == "cuda" else [],\n                           early_stopping=asdict(stopper), best_episode=best_episode, last_loss=last_loss,\n                           sampling_stats=sampling_stats)\n        return payload\n\n    print(f"Algorithm: {config.algorithm} | Device: {device} | At least {config.min_episodes} training episodes | Folder: {run_dir}", flush=True)\n    print("Early-stop patience starts at the first validation after the minimum episode budget.", flush=True)\n    try:\n        for episode in range(completed + 1, config.max_episodes + 1):\n            env = FixedOpponentEnv(config.max_plies, config.learner)\n            position = board_bank.training_position(episode, config.seed) if board_bank is not None else ENDGAMES[config.preset]\n            obs, info = env.reset(position["fen"], f"train:opponent:{config.seed}:{episode}")\n            if info["outcome"] is not None:\n                raise ValueError("The training preset ended before the learner could move.")\n            total_plies += len(env.game.history)\n            episode_steps, reward, epsilon = 0, 0.0, config.epsilon_start\n            while True:\n                state = encode_state(obs)\n                fraction = min(1.0, steps / config.epsilon_decay_steps)\n                epsilon = config.epsilon_start + fraction * (config.epsilon_end - config.epsilon_start)\n                action = (rng.choice(obs["legal_action_ids"]) if rng.random() < epsilon else\n                          greedy_action(model, state, obs["legal_action_ids"], device))\n                next_obs, reward, terminated, truncated, info = env.step(action)\n                next_state = encode_state(next_obs)\n                # A truncated game retains next-state legal IDs for bootstrapping.\n                replay.append((state, action, reward, next_state, terminated,\n                               tuple(next_obs["legal_action_ids"]) if not terminated else ()))\n                steps += 1\n                episode_steps += 1\n                total_plies += info["environment_plies"]\n                if steps >= config.warmup_steps and len(replay) >= config.batch_size and steps % config.train_every == 0:\n                    last_loss = optimize(model, target, optimizer, replay, replay_rng, config, device, sampling_stats)\n                    updates += 1\n                    if updates % config.target_every == 0:\n                        target.load_state_dict(model.state_dict())\n                obs = next_obs\n                if terminated or truncated:\n                    break\n            completed = episode\n            board_details = {"board_id": position["id"], "initial_fen": position["fen"]} if board_bank is not None else {}\n            _append_csv(run_dir / "episodes.csv", {"episode": episode, "learner_steps": steps,\n                        "environment_plies": total_plies, "episode_plies": len(env.game.history),\n                        "reward": reward, "reason": info["outcome"]["reason"],\n                        "epsilon": epsilon, "loss": last_loss, "updates": updates, **board_details})\n            if episode % 25 == 0:\n                print(f"Episode {episode}/{config.max_episodes} | steps {steps} | epsilon {epsilon:.3f}", flush=True)\n            if episode % config.eval_every == 0 or episode == config.min_episodes or episode == config.max_episodes:\n                last_validation = evaluate(model, config, device, positions=validation_positions)\n                improved, stop = stopper.update(episode, last_validation["score"])\n                if improved:\n                    best_episode = episode\n                    _save_torch(run_dir / "best.pt", checkpoint("policy", last_validation))\n                _append_csv(run_dir / "validation.csv", {"episode": episode,\n                            **{k: v for k, v in last_validation.items() if k != "per_board"},\n                            "bad_evals": stopper.bad_evals, "best_episode": best_episode})\n                if board_bank is not None:\n                    _write_json(run_dir / f"validation_ep{episode:04d}.json", last_validation)\n                _write_json(run_dir / "example_game.json", env.game.export())\n                _save_torch(run_dir / "last.pt", checkpoint("training", last_validation))\n                _write_json(run_dir / "sampling.json", {"episode": completed, **sampling_stats,\n                            "mode": config.replay_sampling, "terminal_mix": config.terminal_mix,\n                            "importance_weight": "1 / (buffer_size * item_probability); no normalization"})\n                if on_checkpoint is not None:\n                    on_checkpoint(run_dir, improved)\n                print(f"Validation {episode}: W/L/D/T = {last_validation[\'wins\']}/{last_validation[\'losses\']}/"\n                      f"{last_validation[\'draws\']}/{last_validation[\'truncations\']} | score {last_validation[\'score\']:.3f} | "\n                      f"patience {stopper.bad_evals}/{config.patience}", flush=True)\n                if stop:\n                    stop_reason = "early_stopping"\n                    break\n    except KeyboardInterrupt:\n        # last.pt refers only to a completed validation checkpoint. Preserve it;\n        # do not label a partially played episode as completed training.\n        stop_reason = "interrupted"\n        print("Interrupted. Resume from the latest completed last.pt checkpoint.", flush=True)\n    summary = {"algorithm": config.algorithm,\n               "episodes_completed": completed, "minimum_reached": completed >= config.min_episodes,\n               "learner_steps": steps, "environment_plies": total_plies, "updates": updates,\n               "stop_reason": stop_reason, "best_episode": best_episode,\n               "best_validation_score": stopper.best_score if math.isfinite(stopper.best_score) else None,\n               "seconds_this_session": time.monotonic() - started, "run_dir": str(run_dir)}\n    _write_json(run_dir / "summary.json", summary)\n    if best_episode is not None:\n        best = torch.load(run_dir / "best.pt", map_location=device, weights_only=True)\n        model.load_state_dict(best["model"])\n    print(json.dumps(summary, indent=2), flush=True)\n    return {"model": model, "config": config, "device": device, "summary": summary, "run_dir": run_dir}\n\n\ndef export_policy(run_dir, model_id=None, model_name=None):\n    """Create a complete model folder ZIP for the website, without replay memory."""\n    run_dir = Path(run_dir)\n    saved = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=True)\n    preset = saved["config"]["preset"]\n    algorithm = saved["config"].get("algorithm", "dqn")\n    label = "Double DQN" if algorithm == "double_dqn" else "DQN"\n    folder = model_id if model_id is not None else f"{algorithm}_{preset}"\n    if not folder or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in folder):\n        raise ValueError("Use a simple portable model ID.")\n    output = run_dir / f"{folder}_model.zip"\n    manifest = {"schema": "xiangqi-lab.model.v1", "id": model_id or f"custom-{algorithm.replace(\'_\', \'-\')}-{preset}",\n                "name": model_name or f"{label} {preset}", "description": f"Best validation checkpoint, episode {saved[\'episode\']}; trained as {saved[\'config\'][\'learner\']}.",\n                "protocol": "python", "entrypoint": "policy.py", "python": "{python}",\n                "checkpoint": "best.pt", "device": "cpu", "timeout_seconds": 60}\n    readme = (f"# {label} {preset}\\n\\nTrained as {saved[\'config\'][\'learner\']}, selected at episode {saved[\'episode\']}.\\n"\n              "Extract this folder under xiangqi_lab/models/. Install requirements.txt in the Python environment "\n              "that starts the website, or set model.json\'s python path to a separate environment with PyTorch. "\n              "The portable app\'s bundled Python does not include PyTorch.\\n\\n"\n              "Use Models > Refresh list > Test before selecting the model. Legal moves are masked for either "\n              "colour; playing strength for the untrained colour and unseen boards is not established.\\n\\n"\n              "Validation uses the same fixed starting board with separate opponent seeds. "\n              "It is used for checkpoint selection, not a held-out-board generalisation claim.\\n")\n    if saved.get("board_bank_sha256"):\n        readme = readme.replace("Validation uses the same fixed starting board with separate opponent seeds.",\n                                "Validation uses separate starting boards and fixed opponent seeds; test boards are excluded.")\n    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:\n        archive.write(run_dir / "best.pt", f"{folder}/best.pt")\n        archive.write(Path(__file__).with_name("dqn_model.py"), f"{folder}/dqn_model.py")\n        archive.writestr(f"{folder}/policy.py", "from dqn_model import Policy\\n")\n        archive.writestr(f"{folder}/model.json", json.dumps(manifest, indent=2))\n        archive.writestr(f"{folder}/requirements.txt", "torch>=2.6,<3\\n")\n        archive.writestr(f"{folder}/README.md", readme)\n        for name in ("config.json", "summary.json", "baseline.json", "episodes.csv", "validation.csv", "example_game.json", "board_bank.json", "sampling.json"):\n            if (run_dir / name).exists():\n                archive.write(run_dir / name, f"{folder}/training_report/{name}")\n    return output\n', 'board_bank.py': '"""Separate training and validation starts with a reproducible balanced schedule."""\nimport copy\nimport hashlib\nimport json\nimport random\n\nfrom rules import Game, make_fen\n\n\ndef position_key(fen):\n    return " ".join(Game(fen).fen.split()[:2])\n\n\ndef mirror_key(fen):\n    game = Game(fen)\n    return " ".join(make_fen([list(reversed(row)) for row in game.board], game.side).split()[:2])\n\n\nclass BoardBank:\n    def __init__(self, data):\n        self.data = copy.deepcopy(data)\n        if data.get("schema") != "xiangqi-lab.board-bank.v1":\n            raise ValueError("Unknown board bank schema.")\n        self.training = self.data["training"]\n        self.validation = self.data["validation"]\n        if not self.training or not self.validation:\n            raise ValueError("Training and validation boards are both required.")\n        forbidden = set()\n        for fen in self.data.get("excluded_fens", []):\n            forbidden.update([position_key(fen), mirror_key(fen)])\n        seen, names = set(), set()\n        for item in self.training + self.validation:\n            game = Game(item["fen"])\n            key, mirrored = position_key(item["fen"]), mirror_key(item["fen"])\n            if game.side != "red" or game.outcome() is not None:\n                raise ValueError("Use active Red-to-move starts.")\n            if item["id"] in names or key in seen or key in forbidden:\n                raise ValueError("Duplicate, mirrored, overlapping or excluded board.")\n            names.add(item["id"])\n            seen.update([key, mirrored])\n        self.sha256 = hashlib.sha256(json.dumps(self.data, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n    def training_position(self, episode, seed):\n        if episode < 1:\n            raise ValueError("Episode numbers start at one.")\n        cycle, index = divmod(episode - 1, len(self.training))\n        order = list(range(len(self.training)))\n        random.Random(f"{self.sha256}:train-order:{seed}:{cycle}").shuffle(order)\n        return self.training[order[index]]\n'}
SOURCE_HASHES = {'rules.py': '8fc7f6b38578e0d2538c546d8a755a61f787e5e0c5640e4d025e7a362d9c7225', 'training.py': '1575c584754c0516beafa332210eab8249a709b427fd7a351e31aa8e226429d3', 'notebook_tools.py': '552844feeba9103769325966e9ab852f71969bb9a19a28b3076add7d6122e7b0', 'dqn_model.py': '4f34c3a3a5687c9ced81330315ddfde5dd13cd65f8631ec4b520dd7ac139bb78', 'dqn_training.py': '857d6dadf40f51cc4653e9e2e32f3cb00bb155ebe56602efbe7151c4e8178e2c', 'board_bank.py': '621ae1e90804d02c29ef6cea79f1c948f9a0e07d698b2f00a26bb313e22c78c9'}
BOARD_DATA = {'schema': 'xiangqi-lab.board-bank.v1', 'id': 'ROOK20_20261025_S42', 'training': [{'id': 'rook20-training-01', 'fen': '5k3/9/5a3/9/P8/9/4p4/9/9/3K4R w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/4p4/P8/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2a2', 'e8f7', 'a2i2', 'e5e4', 'a4a5', 'e4e3', 'i2i0', 'e9f9']}, 'red_legal_moves': 17, 'black_legal_moves': 6}, {'id': 'rook20-training-02', 'fen': '3k5/R8/3a5/9/9/2P3p2/9/9/3K5/9 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/6p2/9/2P6/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2i2', 'e8d7', 'i2a2', 'e9d9', 'd0d1', 'g6g5', 'a2e2', 'd9d8', 'e2a2', 'g5g4', 'a2a8', 'd8d9']}, 'red_legal_moves': 21, 'black_legal_moves': 4}, {'id': 'rook20-training-03', 'fen': 'R2a5/4k4/9/9/4P4/7p1/9/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/8p/4P4/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2i2', 'e8d7', 'i2i1', 'd7e8', 'd0e0', 'e8d7', 'i1a1', 'i5i4', 'a1a2', 'd7e8', 'a2a9', 'e8d9', 'e4e5', 'i4h4', 'e0d0', 'e9e8']}, 'red_legal_moves': 17, 'black_legal_moves': 6}, {'id': 'rook20-training-04', 'fen': '4k4/9/5a3/1R7/9/6P2/p8/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/p8/9/6P2/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2f2', 'e8f7', 'f2g2', 'e9e8', 'g2e2', 'e8f8', 'e2e1', 'a6a5', 'e1e9', 'a5a4', 'e9d9', 'f8e8', 'd9d7', 'a4b4', 'd7d6', 'b4a4', 'd6c6', 'a4a3', 'c6b6', 'e8e9']}, 'red_legal_moves': 19, 'black_legal_moves': 5}, {'id': 'rook20-training-05', 'fen': '9/3k5/3a5/9/9/2R5P/3p5/9/3K5/9 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/2p6/8P/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2e1', 'c5c4', 'e1e6', 'c4b4', 'e6e2', 'e9f9', 'e2e5', 'b4c4', 'e5e0', 'f9f8', 'e0e4', 'e8d7', 'e4f4', 'f8e8', 'f4f9', 'e8e7', 'f9f4', 'c4c3', 'd0d1', 'e7e8', 'f4a4', 'e8d8', 'a4c4', 'c3d3']}, 'red_legal_moves': 19, 'black_legal_moves': 6}, {'id': 'rook20-training-06', 'fen': '3k5/9/3a5/9/9/P3p4/1R7/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/4p4/9/P8/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2d2', 'e8d7', 'd2d3', 'e9d9', 'd3g3', 'e6e5', 'g3b3', 'e5e4']}, 'red_legal_moves': 20, 'black_legal_moves': 5}, {'id': 'rook20-training-07', 'fen': '3a5/9/4k4/9/9/2P3p2/3R5/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/6p2/2P6/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2e3', 'e9f9', 'e3e7', 'e8d9', 'e7e9', 'f9f8', 'e9e3', 'f8f7', 'e3b3', 'g5g4', 'b3d3', 'f7e7']}, 'red_legal_moves': 18, 'black_legal_moves': 6}, {'id': 'rook20-training-08', 'fen': '5a3/2R6/5k3/9/9/4P3p/9/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/8p/9/4P4/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2c2', 'e9f9', 'd0d1', 'i6i5', 'c2b2', 'f9e9', 'd1d0', 'e8f9', 'b2b4', 'e9e8', 'b4c4', 'i5i4', 'c4c6', 'e8e7', 'c6c8', 'e7f7']}, 'red_legal_moves': 20, 'black_legal_moves': 4}, {'id': 'rook20-training-09', 'fen': '3a5/9/4k4/9/9/6P2/1p7/9/3K5/R8 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/p8/6P2/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2e1', 'e9f9', 'e1e5', 'f9f8', 'e5e2', 'e8d9', 'd0d1', 'f8f7', 'e2e0', 'a5a4', 'e0a0', 'f7f8', 'a0a2', 'f8f7', 'd1d2', 'a4a3', 'a2a0', 'f7e7', 'd2d1', 'a3b3']}, 'red_legal_moves': 20, 'black_legal_moves': 6}, {'id': 'rook20-training-10', 'fen': '9/4a4/3k5/9/9/3p4P/9/7R1/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/2p6/9/8P/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2e0', 'c6c5', 'e0e1', 'c5c4', 'e1h1', 'e8f7', 'h1h4', 'f7e8', 'h4f4', 'e8f9', 'f4f6', 'c4d4', 'f6i6', 'e9d9', 'i6c6', 'f9e8', 'c6i6', 'd9d8', 'i6f6', 'e8d9', 'f6h6', 'd9e8', 'h6h2', 'd8d7']}, 'red_legal_moves': 20, 'black_legal_moves': 6}, {'id': 'rook20-training-11', 'fen': '4k4/4a4/7R1/9/4p4/P8/9/9/9/4K4 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/4p4/P8/9/4R4/9/3K5 w - - 0 1', 'attempt': 2, 'moves': ['e2h2', 'e8f7', 'h2h0', 'e9f9', 'h0h7', 'f9e9', 'd0e0', 'f7e8']}, 'red_legal_moves': 21, 'black_legal_moves': 7}, {'id': 'rook20-training-12', 'fen': '4k4/4a4/9/9/2P6/6p2/4R4/9/9/5K3 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/6p2/9/2P6/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2e4', 'g6g5', 'c4c5', 'e9f9', 'd0e0', 'f9e9', 'e4e1', 'e9f9', 'e1e3', 'f9e9', 'e0f0', 'g5g4']}, 'red_legal_moves': 21, 'black_legal_moves': 4}, {'id': 'rook20-training-13', 'fen': '4k4/9/3a5/9/6R2/4P3p/9/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/8p/4P4/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2a2', 'e8f9', 'a2c2', 'f9e8', 'c2h2', 'e8d7', 'h2h1', 'e9e8', 'h1b1', 'i5i4', 'b1b5', 'e8e9', 'b5h5', 'e9f9', 'h5g5', 'f9e9']}, 'red_legal_moves': 20, 'black_legal_moves': 6}, {'id': 'rook20-training-14', 'fen': '5k3/4a4/9/4R4/9/6P2/2p6/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/p8/9/6P2/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2f2', 'e8d9', 'f2g2', 'a6a5', 'g2g3', 'a5a4', 'g3d3', 'a4a3', 'd3d2', 'a3b3', 'd2d3', 'd9e8', 'd3d4', 'e8d7', 'd4d5', 'd7e8', 'd5d6', 'b3c3', 'd6e6', 'e9f9']}, 'red_legal_moves': 19, 'black_legal_moves': 8}, {'id': 'rook20-training-15', 'fen': '5k3/9/5a3/8P/9/1p7/9/9/9/3K3R1 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/2p6/8P/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['i4i5', 'e9f9', 'e2e0', 'e8d7', 'i5h5', 'c5c4', 'h5i5', 'd7e8', 'e0f0', 'e8f7', 'f0f1', 'c4b4', 'f1f2', 'f9f8', 'f2b2', 'f7e8', 'i5i6', 'b4a4', 'b2e2', 'a4b4', 'e2e0', 'e8f7', 'e0h0', 'f8f9']}, 'red_legal_moves': 17, 'black_legal_moves': 6}, {'id': 'rook20-training-16', 'fen': '5a3/4k4/R8/4p4/9/P8/9/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/4p4/9/P8/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2f2', 'e8f7', 'f2f3', 'f7e8', 'f3f7', 'e8f9', 'f7a7', 'e9e8']}, 'red_legal_moves': 15, 'black_legal_moves': 3}, {'id': 'rook20-training-17', 'fen': '3k5/4a4/9/9/4R4/2P6/7p1/9/9/4K4 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/6p2/2P6/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2g2', 'g5g4', 'd0e0', 'g4g3', 'g2d2', 'g3h3', 'd2i2', 'h3i3', 'i2e2', 'e9d9', 'e2e5', 'i3h3']}, 'red_legal_moves': 18, 'black_legal_moves': 8}, {'id': 'rook20-training-18', 'fen': '5k3/9/3a5/9/9/4P3p/3R5/9/3K5/9 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/8p/9/4P4/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2e0', 'e9f9', 'e0i0', 'e8d9', 'i0i4', 'd9e8', 'i4g4', 'e8d7', 'g4g8', 'i6i5', 'g8g3', 'f9e9', 'd0d1', 'e9f9', 'g3d3', 'i5i4']}, 'red_legal_moves': 17, 'black_legal_moves': 5}, {'id': 'rook20-training-19', 'fen': '5k3/4a4/9/1R7/9/6P2/1p7/9/4K4/9 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/p8/6P2/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2a2', 'e8f7', 'a2a1', 'e9e8', 'd0d1', 'e8e9', 'a1a2', 'f7e8', 'a2b2', 'e9f9', 'b2b1', 'a5a4', 'b1b2', 'a4a3', 'b2b0', 'f9f8', 'd1e1', 'f8f9', 'b0b6', 'a3b3']}, 'red_legal_moves': 18, 'black_legal_moves': 8}, {'id': 'rook20-training-20', 'fen': '3k5/9/3a5/9/8P/2p6/7R1/9/3K5/9 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/2p6/9/8P/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2i2', 'e8d7', 'i2e2', 'e9d9', 'e2e7', 'd9d8', 'i4i5', 'c6c5', 'e7g7', 'd8e8', 'g7g4', 'e8e9', 'g4i4', 'e9e8', 'd0d1', 'e8d8', 'i4h4', 'd8d9', 'h4h3', 'd9e9', 'h3h4', 'c5c4', 'h4h3', 'e9d9']}, 'red_legal_moves': 22, 'black_legal_moves': 5}], 'validation': [{'id': 'rook20-validation-01', 'fen': '9/4k4/5a3/9/4p4/P8/9/9/9/3K1R3 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/4p4/P8/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2g2', 'e8f7', 'g2g1', 'e9f9', 'g1f1', 'f9f8', 'f1f0', 'f8e8']}, 'red_legal_moves': 14, 'black_legal_moves': 4}, {'id': 'rook20-validation-02', 'fen': '5k3/4a4/9/9/8R/2P6/6p2/9/3K5/9 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/6p2/9/2P6/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2e0', 'g6g5', 'e0e5', 'g5g4', 'e5f5', 'g4g3', 'f5c5', 'e8d9', 'd0d1', 'd9e8', 'c5i5', 'e9f9']}, 'red_legal_moves': 21, 'black_legal_moves': 8}, {'id': 'rook20-validation-03', 'fen': '4k4/4a4/9/9/4P4/1R7/8p/9/9/4K4 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/8p/4P4/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2b2', 'e8d9', 'b2b4', 'e9f9', 'd0d1', 'd9e8', 'd1d0', 'i5i4', 'b4b6', 'i4i3', 'd0e0', 'i3h3', 'e4e5', 'h3i3', 'b6b4', 'f9e9']}, 'red_legal_moves': 23, 'black_legal_moves': 8}, {'id': 'rook20-validation-04', 'fen': '9/9/3ak4/9/p6P1/9/9/5R3/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/p8/9/6P2/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2c2', 'e8d7', 'c2d2', 'e9e8', 'd2d5', 'e8e7', 'd5h5', 'a6a5', 'g4g5', 'd7e8', 'h5h7', 'e8f7', 'h7h2', 'f7e8', 'h2g2', 'e8d9', 'g5h5', 'd9e8', 'g2f2', 'e8d7']}, 'red_legal_moves': 21, 'black_legal_moves': 3}, {'id': 'rook20-validation-05', 'fen': '3ak4/9/1R7/9/8P/p8/9/9/3K5/9 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/9/2p6/8P/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['d0e0', 'e9f9', 'e2e6', 'f9e9', 'e6e2', 'e9f9', 'e2a2', 'f9e9', 'a2a5', 'c5c4', 'e0d0', 'e9f9', 'a5b5', 'c4b4', 'd0d1', 'b4c4', 'b5b1', 'e8d9', 'i4i5', 'f9e9', 'b1b9', 'c4b4', 'b9b7', 'b4a4']}, 'red_legal_moves': 21, 'black_legal_moves': 5}, {'id': 'rook20-validation-06', 'fen': '9/5k3/5a3/9/4p4/P3R4/9/9/9/3K5 w - - 0 1', 'generation': {'base_fen': '4k4/4a4/9/4p4/9/P8/9/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'moves': ['e2e3', 'e8f7', 'e3e5', 'e9e8', 'e5e0', 'e6e5', 'e0e4', 'e8f8']}, 'red_legal_moves': 15, 'black_legal_moves': 4}], 'excluded_fens': ['5a3/4k4/9/7R1/9/9/2P1p4/9/9/3K5 w - - 0 1', '3k5/9/5a3/4R4/4p4/9/2P6/9/9/4K4 w - - 0 1', '9/9/1R1ak4/9/9/9/2Pp5/9/9/3K5 w - - 0 1', '9/4k4/3a5/9/9/6R2/2P6/5K3/4p4/9 w - - 0 1', '3k1a3/2C1a4/9/4p4/9/4P4/9/9/4K4/5A3 w - - 0 1', '3a5/4k4/5a1C1/9/4p4/4P4/9/9/9/3AK4 w - - 0 1', '3k5/4a4/5a3/4p4/3P5/9/9/9/4A4/4KC3 w - - 0 1', '4k4/9/5a3/9/2eP5/4p4/9/9/5H3/3AK4 w - - 0 1', '6e2/3k5/5a3/9/3P5/4p4/5H3/9/5K3/3A5 w - - 0 1', '5k3/4a4/4e4/9/9/2P1H4/4p4/9/9/3AK4 w - - 0 1', '4k4/4a4/9/9/4p4/2P6/9/4R4/9/3K5 w - - 0 1', '3aka3/9/9/4p4/9/4P4/2C6/9/9/3AK4 w - - 0 1', '3ak4/9/4e4/9/4p4/2P6/5H3/9/9/3AK4 w - - 0 1', 'rheakaehr/9/1c5c1/p1p1p1p1p/9/9/P1P1P1P1P/1C5C1/9/RHEAKAEHR w - - 0 1'], 'selection': 'Fixed seeded legal non-capture rollouts; no model scores used.', 'schedule': 'Each block of 20 training episodes visits each training start once in shuffled order.', 'limits': 'Valid custom endgames; not certified winning positions or a representative sample of all endgames. Independence refers to starting-board splits, not all reachable states.', 'old_test_protection': 'Rook test starts have a red pawn on rank 3. All new starts have it on rank 4 or higher, and it cannot move backwards. Cannon and horse test material is absent.'}
EXPERIMENT = "ROOK20_TERMINAL50_20261025_S42"
SOURCE_DIR = Path("/content/xiangqi_terminal50_source")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
for name, source in SOURCE_FILES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name]
    path = SOURCE_DIR / name
    if path.exists() and path.read_text() != source:
        raise RuntimeError("Source changed. Start a fresh runtime.")
    path.write_text(source, encoding="utf-8")
    loaded = sys.modules.get(path.stem)
    if loaded is not None and Path(getattr(loaded, "__file__", "")).resolve() != path.resolve():
        raise RuntimeError("Another project snapshot is loaded. Use a fresh runtime.")
sys.path.insert(0, str(SOURCE_DIR))
importlib.invalidate_caches()
from board_bank import BoardBank
from rules import Game, make_fen
from notebook_tools import board_svg
from dqn_training import TrainConfig, train, export_policy
import torch
from IPython.display import HTML, SVG, display
bank = BoardBank(BOARD_DATA)
assert len(bank.training) == 20 and len(bank.validation) == 6
for item in bank.training + bank.validation:
    probe = Game(item["generation"]["base_fen"])
    for move in item["generation"]["moves"]:
        probe.step(move)
    assert make_fen(probe.board, "red") == item["fen"]
BASE_DIR = Path("/content/xiangqi_output") / EXPERIMENT
BASE_DIR.mkdir(parents=True, exist_ok=True)
(BASE_DIR / "boards.json").write_text(json.dumps(BOARD_DATA, indent=2))
(BASE_DIR / "source_hashes.json").write_text(json.dumps(SOURCE_HASHES, indent=2))
print("20 training boards; 6 validation boards; old ten test starts excluded.")
print("Board bank SHA256:", bank.sha256)
print("PyTorch:", torch.__version__, "| Device: CPU")
print("No training has started.")

## 2. Set the run

Sets the episode limit, validation schedule and early stopping. Leave training off when only checking the notebook.

In [ ]:
# @title Training settings
START_TRAINING = False  # @param {type:"boolean"}
RESUME = False  # @param {type:"boolean"}
RUNS = [("dqn", "DQN_ROOK20_TERMINAL50_20261025_S42"),
        ("double_dqn", "DoubleDQN_ROOK20_TERMINAL50_20261025_S42")]
CONFIGS = {algorithm: TrainConfig(
    algorithm=algorithm, preset="rook", learner="red", min_episodes=1000,
    max_episodes=3000, max_plies=160, eval_every=100, eval_games=60,
    patience=5, min_delta=0.02, seed=42, epsilon_decay_steps=100000, device="cpu",
    replay_sampling="terminal_mixture", terminal_mix=0.5
) for algorithm, _ in RUNS}
for config in CONFIGS.values():
    config.validate()
print("Each model: 1000 to 3000 episodes; check every 100 episodes.")
print("Each check: 6 validation boards x 10 seeds = 60 games.")
print("Stop after 5 checks without a gain above 0.02, after at least 1000 episodes.")
print("Balanced training-board order and the same opponent seeds for both models.")
print("Fresh seed-42 initialization; no old checkpoint is loaded.")
print("Replay: 50% uniform; 50% terminal win/loss/draw groups; exact importance weights.")


## 3. Train and compare

Train both models, save their best checkpoints, then test each on the same 200 cases without learning.


In [ ]:
# @title Train both models and keep backups
import base64, csv, io, zipfile
from contextlib import redirect_stdout

class Tee:
    def __init__(self, output, file):
        self.output, self.file = output, file
    def write(self, text):
        self.file.write(text)
        self.file.flush()
        return self.output.write(text)
    def flush(self):
        self.output.flush()
        self.file.flush()

def launch_rook20():
    if not START_TRAINING:
        print("Training is off. Check START_TRAINING to begin.")
        return
    backups = {}
    view = display(HTML("<p>Waiting for the first best checkpoint.</p>"), display_id=True)
    def publish(run_dir, run_name, algorithm):
        policy_id = "custom-" + run_name.lower().replace("_", "-")
        package = export_policy(run_dir, model_id=policy_id, model_name=run_name)
        saved = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=True)
        label = f"{run_name}_BEST_EP{saved['episode']:04d}.zip"
        path = BASE_DIR / label
        path.write_bytes(package.read_bytes())
        meta = {"run": run_name, "algorithm": algorithm, "best_episode": saved["episode"],
                "checkpoint_sha256": hashlib.sha256((run_dir / "best.pt").read_bytes()).hexdigest(),
                "zip_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                "validation": saved["validation"], "board_bank_sha256": bank.sha256}
        (BASE_DIR / f"{run_name}_backup.json").write_text(json.dumps(meta, indent=2))
        backups[run_name] = (path, meta)
        blocks = ["<h3>Saved best models</h3><p>New experiment only. Old model files are unchanged.</p>"]
        for name, (file, info) in backups.items():
            encoded = base64.b64encode(file.read_bytes()).decode("ascii")
            blocks.append(f'<p><b>{name}</b> | best episode {info["best_episode"]} | validation {info["validation"]["wins"]}/60 wins<br>'
                          f'<a download="{file.name}" data-sha256="{info["zip_sha256"]}" href="data:application/zip;base64,{encoded}">Save {name}</a><br>'
                          f'<small>Checkpoint SHA256: {info["checkpoint_sha256"]}</small></p>')
        view.update(HTML("".join(blocks)))
        print("BACKUP", label, "checkpoint_sha256", meta["checkpoint_sha256"], flush=True)

    for algorithm, run_name in RUNS:
        folder = BASE_DIR / run_name
        done = folder / "summary.json"
        if done.exists():
            summary = json.loads(done.read_text())
            if summary["stop_reason"] in ("early_stopping", "max_episodes"):
                saved = torch.load(folder / "best.pt", map_location="cpu", weights_only=True)
                assert saved["config"] == vars(CONFIGS[algorithm])
                assert saved["board_bank_sha256"] == bank.sha256
                assert saved["source_hashes"] == SOURCE_HASHES
                publish(folder, run_name, algorithm)
                print("Already complete:", run_name, "| No training repeated.", flush=True)
                continue
        resume = bool(RESUME and (folder / "last.pt").exists())
        def on_checkpoint(path, improved):
            if improved:
                publish(path, run_name, algorithm)
        with (BASE_DIR / f"{run_name}.log").open("a", encoding="utf-8") as log:
            with redirect_stdout(Tee(sys.stdout, log)):
                print("START", run_name, "| Resume:", resume, flush=True)
                result = train(CONFIGS[algorithm], folder, resume=resume, source_hashes=SOURCE_HASHES,
                               board_bank=bank, on_checkpoint=on_checkpoint)
                if result["summary"]["stop_reason"] == "interrupted":
                    print("Stopped. Enable RESUME to continue this run.", flush=True)
                    return
                publish(folder, run_name, algorithm)
    summaries = {name: json.loads((BASE_DIR / name / "summary.json").read_text()) for _, name in RUNS}
    (BASE_DIR / "completed.json").write_text(json.dumps({"status": "complete", "runs": summaries,
        "board_bank_sha256": bank.sha256, "test_games_run": 0}, indent=2))
    print("BOTH RUNS COMPLETE. Best models are saved. No test boards were used.", flush=True)
    for name, info in summaries.items():
        print(name, "| trained", info["episodes_completed"], "| best episode", info["best_episode"],
              "| best validation score", info["best_validation_score"], flush=True)

launch_rook20()

if START_TRAINING and (BASE_DIR / "completed.json").is_file():
    # Tests start only after the training and best-model selection are complete.
    TEST_DATA = {'split': 'unseen-10-20261025-v1', 'created_utc': '2026-10-25T00:58:54.530767+00:00', 'board_count': 10, 'games_per_board_per_model': 20, 'boards': [{'id': 'rook-01', 'group': 'rook', 'fen': '5a3/4k4/9/7R1/9/9/2P1p4/9/9/3K5 w - - 0 1', 'piece_count': 6, 'red_legal_moves': 20, 'black_legal_moves': 6, 'unseen_evidence': 'Red pawn on rank 3; training pawn started on rank 4 and cannot return to rank 3.', 'generation': {'base_fen': '4k4/4a4/9/9/4p4/9/2P6/4R4/9/3K5 w - - 0 1', 'attempt': 2, 'legal_moves': ['e2i2', 'e8f9', 'i2d2', 'f9e8', 'd2h2', 'e8f9', 'h2h9', 'e5e4', 'h9h7', 'e9e8', 'h7h6', 'e4e3'], 'seed': 'unseen-10-20261025-v1:board:rook-01'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:rook-01:00', 'unseen-10-20261025-v1:opponent:rook-01:01', 'unseen-10-20261025-v1:opponent:rook-01:02', 'unseen-10-20261025-v1:opponent:rook-01:03', 'unseen-10-20261025-v1:opponent:rook-01:04', 'unseen-10-20261025-v1:opponent:rook-01:05', 'unseen-10-20261025-v1:opponent:rook-01:06', 'unseen-10-20261025-v1:opponent:rook-01:07', 'unseen-10-20261025-v1:opponent:rook-01:08', 'unseen-10-20261025-v1:opponent:rook-01:09', 'unseen-10-20261025-v1:opponent:rook-01:10', 'unseen-10-20261025-v1:opponent:rook-01:11', 'unseen-10-20261025-v1:opponent:rook-01:12', 'unseen-10-20261025-v1:opponent:rook-01:13', 'unseen-10-20261025-v1:opponent:rook-01:14', 'unseen-10-20261025-v1:opponent:rook-01:15', 'unseen-10-20261025-v1:opponent:rook-01:16', 'unseen-10-20261025-v1:opponent:rook-01:17', 'unseen-10-20261025-v1:opponent:rook-01:18', 'unseen-10-20261025-v1:opponent:rook-01:19']}, {'id': 'rook-02', 'group': 'rook', 'fen': '3k5/9/5a3/4R4/4p4/9/2P6/9/9/4K4 w - - 0 1', 'piece_count': 6, 'red_legal_moves': 15, 'black_legal_moves': 3, 'unseen_evidence': 'Red pawn on rank 3; training pawn started on rank 4 and cannot return to rank 3.', 'generation': {'base_fen': '4k4/4a4/9/9/4p4/9/2P6/4R4/9/3K5 w - - 0 1', 'attempt': 1, 'legal_moves': ['e2f2', 'e8f7', 'f2a2', 'e9e8', 'a2a6', 'e8f8', 'a6e6', 'f8f9', 'e6c6', 'f9e9', 'c6f6', 'e9f9', 'd0e0', 'f9e9', 'f6e6', 'e9d9'], 'seed': 'unseen-10-20261025-v1:board:rook-02'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:rook-02:00', 'unseen-10-20261025-v1:opponent:rook-02:01', 'unseen-10-20261025-v1:opponent:rook-02:02', 'unseen-10-20261025-v1:opponent:rook-02:03', 'unseen-10-20261025-v1:opponent:rook-02:04', 'unseen-10-20261025-v1:opponent:rook-02:05', 'unseen-10-20261025-v1:opponent:rook-02:06', 'unseen-10-20261025-v1:opponent:rook-02:07', 'unseen-10-20261025-v1:opponent:rook-02:08', 'unseen-10-20261025-v1:opponent:rook-02:09', 'unseen-10-20261025-v1:opponent:rook-02:10', 'unseen-10-20261025-v1:opponent:rook-02:11', 'unseen-10-20261025-v1:opponent:rook-02:12', 'unseen-10-20261025-v1:opponent:rook-02:13', 'unseen-10-20261025-v1:opponent:rook-02:14', 'unseen-10-20261025-v1:opponent:rook-02:15', 'unseen-10-20261025-v1:opponent:rook-02:16', 'unseen-10-20261025-v1:opponent:rook-02:17', 'unseen-10-20261025-v1:opponent:rook-02:18', 'unseen-10-20261025-v1:opponent:rook-02:19']}, {'id': 'rook-03', 'group': 'rook', 'fen': '9/9/1R1ak4/9/9/9/2Pp5/9/9/3K5 w - - 0 1', 'piece_count': 6, 'red_legal_moves': 14, 'black_legal_moves': 5, 'unseen_evidence': 'Red pawn on rank 3; training pawn started on rank 4 and cannot return to rank 3.', 'generation': {'base_fen': '4k4/4a4/9/9/4p4/9/2P6/4R4/9/3K5 w - - 0 1', 'attempt': 2, 'legal_moves': ['e2b2', 'e8d9', 'b2d2', 'e5e4', 'd2b2', 'e9e8', 'b2b4', 'e4f4', 'b4b0', 'f4f3', 'b0b6', 'f3e3', 'b6b3', 'e8e7', 'b3b8', 'e3d3', 'b8b1', 'd9e8', 'b1b7', 'e8d7'], 'seed': 'unseen-10-20261025-v1:board:rook-03'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:rook-03:00', 'unseen-10-20261025-v1:opponent:rook-03:01', 'unseen-10-20261025-v1:opponent:rook-03:02', 'unseen-10-20261025-v1:opponent:rook-03:03', 'unseen-10-20261025-v1:opponent:rook-03:04', 'unseen-10-20261025-v1:opponent:rook-03:05', 'unseen-10-20261025-v1:opponent:rook-03:06', 'unseen-10-20261025-v1:opponent:rook-03:07', 'unseen-10-20261025-v1:opponent:rook-03:08', 'unseen-10-20261025-v1:opponent:rook-03:09', 'unseen-10-20261025-v1:opponent:rook-03:10', 'unseen-10-20261025-v1:opponent:rook-03:11', 'unseen-10-20261025-v1:opponent:rook-03:12', 'unseen-10-20261025-v1:opponent:rook-03:13', 'unseen-10-20261025-v1:opponent:rook-03:14', 'unseen-10-20261025-v1:opponent:rook-03:15', 'unseen-10-20261025-v1:opponent:rook-03:16', 'unseen-10-20261025-v1:opponent:rook-03:17', 'unseen-10-20261025-v1:opponent:rook-03:18', 'unseen-10-20261025-v1:opponent:rook-03:19']}, {'id': 'rook-04', 'group': 'rook', 'fen': '9/4k4/3a5/9/9/6R2/2P6/5K3/4p4/9 w - - 0 1', 'piece_count': 6, 'red_legal_moves': 18, 'black_legal_moves': 6, 'unseen_evidence': 'Red pawn on rank 3; training pawn started on rank 4 and cannot return to rank 3.', 'generation': {'base_fen': '4k4/4a4/9/9/4p4/9/2P6/4R4/9/3K5 w - - 0 1', 'attempt': 2, 'legal_moves': ['e2f2', 'e8d7', 'd0d1', 'e5e4', 'd1e1', 'e9e8', 'e1e2', 'e8d8', 'e2e1', 'e4e3', 'f2i2', 'd8e8', 'i2g2', 'e3e2', 'e1f1', 'e8d8', 'g2g9', 'e2e1', 'f1f2', 'e1f1', 'g9g1', 'd8e8', 'g1g4', 'f1e1'], 'seed': 'unseen-10-20261025-v1:board:rook-04'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:rook-04:00', 'unseen-10-20261025-v1:opponent:rook-04:01', 'unseen-10-20261025-v1:opponent:rook-04:02', 'unseen-10-20261025-v1:opponent:rook-04:03', 'unseen-10-20261025-v1:opponent:rook-04:04', 'unseen-10-20261025-v1:opponent:rook-04:05', 'unseen-10-20261025-v1:opponent:rook-04:06', 'unseen-10-20261025-v1:opponent:rook-04:07', 'unseen-10-20261025-v1:opponent:rook-04:08', 'unseen-10-20261025-v1:opponent:rook-04:09', 'unseen-10-20261025-v1:opponent:rook-04:10', 'unseen-10-20261025-v1:opponent:rook-04:11', 'unseen-10-20261025-v1:opponent:rook-04:12', 'unseen-10-20261025-v1:opponent:rook-04:13', 'unseen-10-20261025-v1:opponent:rook-04:14', 'unseen-10-20261025-v1:opponent:rook-04:15', 'unseen-10-20261025-v1:opponent:rook-04:16', 'unseen-10-20261025-v1:opponent:rook-04:17', 'unseen-10-20261025-v1:opponent:rook-04:18', 'unseen-10-20261025-v1:opponent:rook-04:19']}, {'id': 'cannon-01', 'group': 'cannon', 'fen': '3k1a3/2C1a4/9/4p4/9/4P4/9/9/4K4/5A3 w - - 0 1', 'piece_count': 8, 'red_legal_moves': 16, 'black_legal_moves': 5, 'unseen_evidence': 'Contains a red C; this piece was absent from every training position.', 'generation': {'base_fen': '3aka3/9/9/4p4/9/4P4/2C6/9/9/3AK4 w - - 0 1', 'attempt': 1, 'legal_moves': ['d0e1', 'd9e8', 'c3c8', 'e9d9', 'e1f0', 'e8f7', 'c8c9', 'd9e9', 'c9c8', 'f7e8', 'e0e1', 'e9d9'], 'seed': 'unseen-10-20261025-v1:board:cannon-01'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:cannon-01:00', 'unseen-10-20261025-v1:opponent:cannon-01:01', 'unseen-10-20261025-v1:opponent:cannon-01:02', 'unseen-10-20261025-v1:opponent:cannon-01:03', 'unseen-10-20261025-v1:opponent:cannon-01:04', 'unseen-10-20261025-v1:opponent:cannon-01:05', 'unseen-10-20261025-v1:opponent:cannon-01:06', 'unseen-10-20261025-v1:opponent:cannon-01:07', 'unseen-10-20261025-v1:opponent:cannon-01:08', 'unseen-10-20261025-v1:opponent:cannon-01:09', 'unseen-10-20261025-v1:opponent:cannon-01:10', 'unseen-10-20261025-v1:opponent:cannon-01:11', 'unseen-10-20261025-v1:opponent:cannon-01:12', 'unseen-10-20261025-v1:opponent:cannon-01:13', 'unseen-10-20261025-v1:opponent:cannon-01:14', 'unseen-10-20261025-v1:opponent:cannon-01:15', 'unseen-10-20261025-v1:opponent:cannon-01:16', 'unseen-10-20261025-v1:opponent:cannon-01:17', 'unseen-10-20261025-v1:opponent:cannon-01:18', 'unseen-10-20261025-v1:opponent:cannon-01:19']}, {'id': 'cannon-02', 'group': 'cannon', 'fen': '3a5/4k4/5a1C1/9/4p4/4P4/9/9/9/3AK4 w - - 0 1', 'piece_count': 8, 'red_legal_moves': 15, 'black_legal_moves': 4, 'unseen_evidence': 'Contains a red C; this piece was absent from every training position.', 'generation': {'base_fen': '3aka3/9/9/4p4/9/4P4/2C6/9/9/3AK4 w - - 0 1', 'attempt': 1, 'legal_moves': ['c3h3', 'f9e8', 'h3a3', 'e8f7', 'a3g3', 'e9f9', 'g3g0', 'd9e8', 'g0g2', 'e6e5', 'g2g6', 'e8d9', 'g6g7', 'f9e9', 'g7h7', 'e9e8'], 'seed': 'unseen-10-20261025-v1:board:cannon-02'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:cannon-02:00', 'unseen-10-20261025-v1:opponent:cannon-02:01', 'unseen-10-20261025-v1:opponent:cannon-02:02', 'unseen-10-20261025-v1:opponent:cannon-02:03', 'unseen-10-20261025-v1:opponent:cannon-02:04', 'unseen-10-20261025-v1:opponent:cannon-02:05', 'unseen-10-20261025-v1:opponent:cannon-02:06', 'unseen-10-20261025-v1:opponent:cannon-02:07', 'unseen-10-20261025-v1:opponent:cannon-02:08', 'unseen-10-20261025-v1:opponent:cannon-02:09', 'unseen-10-20261025-v1:opponent:cannon-02:10', 'unseen-10-20261025-v1:opponent:cannon-02:11', 'unseen-10-20261025-v1:opponent:cannon-02:12', 'unseen-10-20261025-v1:opponent:cannon-02:13', 'unseen-10-20261025-v1:opponent:cannon-02:14', 'unseen-10-20261025-v1:opponent:cannon-02:15', 'unseen-10-20261025-v1:opponent:cannon-02:16', 'unseen-10-20261025-v1:opponent:cannon-02:17', 'unseen-10-20261025-v1:opponent:cannon-02:18', 'unseen-10-20261025-v1:opponent:cannon-02:19']}, {'id': 'cannon-03', 'group': 'cannon', 'fen': '3k5/4a4/5a3/4p4/3P5/9/9/9/4A4/4KC3 w - - 0 1', 'piece_count': 8, 'red_legal_moves': 16, 'black_legal_moves': 5, 'unseen_evidence': 'Contains a red C; this piece was absent from every training position.', 'generation': {'base_fen': '3aka3/9/9/4p4/9/4P4/2C6/9/9/3AK4 w - - 0 1', 'attempt': 1, 'legal_moves': ['e4e5', 'd9e8', 'c3c9', 'e8f7', 'c9d9', 'e9e8', 'd9d1', 'e8f8', 'd1b1', 'f8e8', 'b1g1', 'e8e9', 'g1f1', 'f9e8', 'e5d5', 'e9d9', 'f1f0', 'e8d7', 'd0e1', 'd7e8'], 'seed': 'unseen-10-20261025-v1:board:cannon-03'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:cannon-03:00', 'unseen-10-20261025-v1:opponent:cannon-03:01', 'unseen-10-20261025-v1:opponent:cannon-03:02', 'unseen-10-20261025-v1:opponent:cannon-03:03', 'unseen-10-20261025-v1:opponent:cannon-03:04', 'unseen-10-20261025-v1:opponent:cannon-03:05', 'unseen-10-20261025-v1:opponent:cannon-03:06', 'unseen-10-20261025-v1:opponent:cannon-03:07', 'unseen-10-20261025-v1:opponent:cannon-03:08', 'unseen-10-20261025-v1:opponent:cannon-03:09', 'unseen-10-20261025-v1:opponent:cannon-03:10', 'unseen-10-20261025-v1:opponent:cannon-03:11', 'unseen-10-20261025-v1:opponent:cannon-03:12', 'unseen-10-20261025-v1:opponent:cannon-03:13', 'unseen-10-20261025-v1:opponent:cannon-03:14', 'unseen-10-20261025-v1:opponent:cannon-03:15', 'unseen-10-20261025-v1:opponent:cannon-03:16', 'unseen-10-20261025-v1:opponent:cannon-03:17', 'unseen-10-20261025-v1:opponent:cannon-03:18', 'unseen-10-20261025-v1:opponent:cannon-03:19']}, {'id': 'horse-01', 'group': 'horse', 'fen': '4k4/9/5a3/9/2eP5/4p4/9/9/5H3/3AK4 w - - 0 1', 'piece_count': 8, 'red_legal_moves': 11, 'black_legal_moves': 7, 'unseen_evidence': 'Contains a red H; this piece was absent from every training position.', 'generation': {'base_fen': '3ak4/9/4e4/9/4p4/2P6/5H3/9/9/3AK4 w - - 0 1', 'attempt': 1, 'legal_moves': ['c4c5', 'e7c9', 'c5d5', 'd9e8', 'f3d4', 'c9a7', 'd4f5', 'e8f7', 'f5g3', 'e5e4', 'g3f1', 'a7c5'], 'seed': 'unseen-10-20261025-v1:board:horse-01'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:horse-01:00', 'unseen-10-20261025-v1:opponent:horse-01:01', 'unseen-10-20261025-v1:opponent:horse-01:02', 'unseen-10-20261025-v1:opponent:horse-01:03', 'unseen-10-20261025-v1:opponent:horse-01:04', 'unseen-10-20261025-v1:opponent:horse-01:05', 'unseen-10-20261025-v1:opponent:horse-01:06', 'unseen-10-20261025-v1:opponent:horse-01:07', 'unseen-10-20261025-v1:opponent:horse-01:08', 'unseen-10-20261025-v1:opponent:horse-01:09', 'unseen-10-20261025-v1:opponent:horse-01:10', 'unseen-10-20261025-v1:opponent:horse-01:11', 'unseen-10-20261025-v1:opponent:horse-01:12', 'unseen-10-20261025-v1:opponent:horse-01:13', 'unseen-10-20261025-v1:opponent:horse-01:14', 'unseen-10-20261025-v1:opponent:horse-01:15', 'unseen-10-20261025-v1:opponent:horse-01:16', 'unseen-10-20261025-v1:opponent:horse-01:17', 'unseen-10-20261025-v1:opponent:horse-01:18', 'unseen-10-20261025-v1:opponent:horse-01:19']}, {'id': 'horse-02', 'group': 'horse', 'fen': '6e2/3k5/5a3/9/3P5/4p4/5H3/9/5K3/3A5 w - - 0 1', 'piece_count': 8, 'red_legal_moves': 15, 'black_legal_moves': 9, 'unseen_evidence': 'Contains a red H; this piece was absent from every training position.', 'generation': {'base_fen': '3ak4/9/4e4/9/4p4/2P6/5H3/9/9/3AK4 w - - 0 1', 'attempt': 1, 'legal_moves': ['f3g5', 'd9e8', 'c4c5', 'e7g9', 'e0e1', 'g9e7', 'e1f1', 'e8f7', 'g5f3', 'e7g9', 'f3h2', 'e9e8', 'c5d5', 'e5e4', 'h2f3', 'e8d8'], 'seed': 'unseen-10-20261025-v1:board:horse-02'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:horse-02:00', 'unseen-10-20261025-v1:opponent:horse-02:01', 'unseen-10-20261025-v1:opponent:horse-02:02', 'unseen-10-20261025-v1:opponent:horse-02:03', 'unseen-10-20261025-v1:opponent:horse-02:04', 'unseen-10-20261025-v1:opponent:horse-02:05', 'unseen-10-20261025-v1:opponent:horse-02:06', 'unseen-10-20261025-v1:opponent:horse-02:07', 'unseen-10-20261025-v1:opponent:horse-02:08', 'unseen-10-20261025-v1:opponent:horse-02:09', 'unseen-10-20261025-v1:opponent:horse-02:10', 'unseen-10-20261025-v1:opponent:horse-02:11', 'unseen-10-20261025-v1:opponent:horse-02:12', 'unseen-10-20261025-v1:opponent:horse-02:13', 'unseen-10-20261025-v1:opponent:horse-02:14', 'unseen-10-20261025-v1:opponent:horse-02:15', 'unseen-10-20261025-v1:opponent:horse-02:16', 'unseen-10-20261025-v1:opponent:horse-02:17', 'unseen-10-20261025-v1:opponent:horse-02:18', 'unseen-10-20261025-v1:opponent:horse-02:19']}, {'id': 'horse-03', 'group': 'horse', 'fen': '5k3/4a4/4e4/9/9/2P1H4/4p4/9/9/3AK4 w - - 0 1', 'piece_count': 8, 'red_legal_moves': 9, 'black_legal_moves': 12, 'unseen_evidence': 'Contains a red H; this piece was absent from every training position.', 'generation': {'base_fen': '3ak4/9/4e4/9/4p4/2P6/5H3/9/9/3AK4 w - - 0 1', 'attempt': 1, 'legal_moves': ['f3h2', 'e7g5', 'h2i4', 'e9f9', 'i4g3', 'e5e4', 'd0e1', 'e4e3', 'g3e4', 'e3d3', 'e4c5', 'f9f8', 'c5a6', 'f8f9', 'e1d0', 'd3e3', 'a6c5', 'g5e7', 'c5e4', 'd9e8'], 'seed': 'unseen-10-20261025-v1:board:horse-03'}, 'opponent_seeds': ['unseen-10-20261025-v1:opponent:horse-03:00', 'unseen-10-20261025-v1:opponent:horse-03:01', 'unseen-10-20261025-v1:opponent:horse-03:02', 'unseen-10-20261025-v1:opponent:horse-03:03', 'unseen-10-20261025-v1:opponent:horse-03:04', 'unseen-10-20261025-v1:opponent:horse-03:05', 'unseen-10-20261025-v1:opponent:horse-03:06', 'unseen-10-20261025-v1:opponent:horse-03:07', 'unseen-10-20261025-v1:opponent:horse-03:08', 'unseen-10-20261025-v1:opponent:horse-03:09', 'unseen-10-20261025-v1:opponent:horse-03:10', 'unseen-10-20261025-v1:opponent:horse-03:11', 'unseen-10-20261025-v1:opponent:horse-03:12', 'unseen-10-20261025-v1:opponent:horse-03:13', 'unseen-10-20261025-v1:opponent:horse-03:14', 'unseen-10-20261025-v1:opponent:horse-03:15', 'unseen-10-20261025-v1:opponent:horse-03:16', 'unseen-10-20261025-v1:opponent:horse-03:17', 'unseen-10-20261025-v1:opponent:horse-03:18', 'unseen-10-20261025-v1:opponent:horse-03:19']}], 'selection': 'Deterministic legal non-capture rollouts; no model scores used in board selection.', 'checks': ['Valid piece counts and locations', 'Neither side starts in check', 'At least 6 Red and 3 Black legal moves', 'No immediate Red win', 'Unique boards and no horizontal-mirror duplicates', 'Not any website preset', 'Unseen even among states reachable from the training start'], 'limits': 'Custom legal endgames, not certified wins or a representative random sample of all Xiangqi endgames. Legal rollouts start from custom valid positions, not from a recorded full game. History and counters reset for each puzzle.'}
    REFERENCE = {'status': 'complete', 'training': False, 'epsilon': 0, 'device': 'cpu', 'results': {'old_dqn': {'wins': 23, 'losses': 0, 'draws': 170, 'truncations': 7, 'games': 200, 'win_rate': 0.115, 'reasons': {'repetition': 161, 'stalemate': 20, 'ply_limit': 7, 'checkmate': 3, 'no_capture': 9}, 'label': 'Old DQN', 'best_episode': 100, 'weights_unchanged': True, 'checkpoint_sha256': 'b6ad0a3b60d4cde7846a887c33c0a60835710b617d9aaf0a131ea1b98cf8c037', 'per_group': {'rook': {'wins': 20, 'losses': 0, 'draws': 60, 'truncations': 0, 'games': 80, 'win_rate': 0.25, 'reasons': {'repetition': 60, 'stalemate': 20}}, 'cannon': {'wins': 3, 'losses': 0, 'draws': 53, 'truncations': 4, 'games': 60, 'win_rate': 0.05, 'reasons': {'repetition': 50, 'ply_limit': 4, 'checkmate': 3, 'no_capture': 3}}, 'horse': {'wins': 0, 'losses': 0, 'draws': 57, 'truncations': 3, 'games': 60, 'win_rate': 0.0, 'reasons': {'no_capture': 6, 'repetition': 51, 'ply_limit': 3}}}, 'per_board': {'rook-01': {'wins': 3, 'losses': 0, 'draws': 17, 'truncations': 0, 'games': 20, 'win_rate': 0.15, 'reasons': {'repetition': 17, 'stalemate': 3}}, 'rook-02': {'wins': 15, 'losses': 0, 'draws': 5, 'truncations': 0, 'games': 20, 'win_rate': 0.75, 'reasons': {'stalemate': 15, 'repetition': 5}}, 'rook-03': {'wins': 2, 'losses': 0, 'draws': 18, 'truncations': 0, 'games': 20, 'win_rate': 0.1, 'reasons': {'repetition': 18, 'stalemate': 2}}, 'rook-04': {'wins': 0, 'losses': 0, 'draws': 20, 'truncations': 0, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 20}}, 'cannon-01': {'wins': 1, 'losses': 0, 'draws': 16, 'truncations': 3, 'games': 20, 'win_rate': 0.05, 'reasons': {'repetition': 16, 'ply_limit': 3, 'checkmate': 1}}, 'cannon-02': {'wins': 2, 'losses': 0, 'draws': 17, 'truncations': 1, 'games': 20, 'win_rate': 0.1, 'reasons': {'repetition': 16, 'no_capture': 1, 'checkmate': 2, 'ply_limit': 1}}, 'cannon-03': {'wins': 0, 'losses': 0, 'draws': 20, 'truncations': 0, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 18, 'no_capture': 2}}, 'horse-01': {'wins': 0, 'losses': 0, 'draws': 20, 'truncations': 0, 'games': 20, 'win_rate': 0.0, 'reasons': {'no_capture': 4, 'repetition': 16}}, 'horse-02': {'wins': 0, 'losses': 0, 'draws': 19, 'truncations': 1, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 17, 'ply_limit': 1, 'no_capture': 2}}, 'horse-03': {'wins': 0, 'losses': 0, 'draws': 18, 'truncations': 2, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 18, 'ply_limit': 2}}}}, 'new_dqn': {'wins': 25, 'losses': 1, 'draws': 168, 'truncations': 6, 'games': 200, 'win_rate': 0.125, 'reasons': {'repetition': 159, 'checkmate': 23, 'no_capture': 9, 'stalemate': 3, 'ply_limit': 6}, 'label': 'New DQN (20 boards)', 'best_episode': 100, 'weights_unchanged': True, 'checkpoint_sha256': '7af30e2b6bc4ad88b7d2c8fd89c44f8130daa1d22be8bb90be7618bc1b25c219', 'per_group': {'rook': {'wins': 17, 'losses': 0, 'draws': 63, 'truncations': 0, 'games': 80, 'win_rate': 0.2125, 'reasons': {'repetition': 62, 'checkmate': 16, 'no_capture': 1, 'stalemate': 1}}, 'cannon': {'wins': 8, 'losses': 1, 'draws': 50, 'truncations': 1, 'games': 60, 'win_rate': 0.13333333333333333, 'reasons': {'repetition': 45, 'stalemate': 2, 'checkmate': 7, 'no_capture': 5, 'ply_limit': 1}}, 'horse': {'wins': 0, 'losses': 0, 'draws': 55, 'truncations': 5, 'games': 60, 'win_rate': 0.0, 'reasons': {'repetition': 52, 'ply_limit': 5, 'no_capture': 3}}}, 'per_board': {'rook-01': {'wins': 4, 'losses': 0, 'draws': 16, 'truncations': 0, 'games': 20, 'win_rate': 0.2, 'reasons': {'repetition': 16, 'checkmate': 4}}, 'rook-02': {'wins': 2, 'losses': 0, 'draws': 18, 'truncations': 0, 'games': 20, 'win_rate': 0.1, 'reasons': {'repetition': 18, 'checkmate': 2}}, 'rook-03': {'wins': 6, 'losses': 0, 'draws': 14, 'truncations': 0, 'games': 20, 'win_rate': 0.3, 'reasons': {'repetition': 13, 'checkmate': 5, 'no_capture': 1, 'stalemate': 1}}, 'rook-04': {'wins': 5, 'losses': 0, 'draws': 15, 'truncations': 0, 'games': 20, 'win_rate': 0.25, 'reasons': {'repetition': 15, 'checkmate': 5}}, 'cannon-01': {'wins': 4, 'losses': 0, 'draws': 16, 'truncations': 0, 'games': 20, 'win_rate': 0.2, 'reasons': {'repetition': 16, 'stalemate': 1, 'checkmate': 3}}, 'cannon-02': {'wins': 2, 'losses': 0, 'draws': 17, 'truncations': 1, 'games': 20, 'win_rate': 0.1, 'reasons': {'checkmate': 2, 'repetition': 13, 'no_capture': 4, 'ply_limit': 1}}, 'cannon-03': {'wins': 2, 'losses': 1, 'draws': 17, 'truncations': 0, 'games': 20, 'win_rate': 0.1, 'reasons': {'repetition': 16, 'checkmate': 2, 'no_capture': 1, 'stalemate': 1}}, 'horse-01': {'wins': 0, 'losses': 0, 'draws': 19, 'truncations': 1, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 16, 'ply_limit': 1, 'no_capture': 3}}, 'horse-02': {'wins': 0, 'losses': 0, 'draws': 18, 'truncations': 2, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 18, 'ply_limit': 2}}, 'horse-03': {'wins': 0, 'losses': 0, 'draws': 18, 'truncations': 2, 'games': 20, 'win_rate': 0.0, 'reasons': {'ply_limit': 2, 'repetition': 18}}}}, 'old_double_dqn': {'wins': 33, 'losses': 0, 'draws': 158, 'truncations': 9, 'games': 200, 'win_rate': 0.165, 'reasons': {'repetition': 138, 'checkmate': 31, 'stalemate': 2, 'no_capture': 20, 'ply_limit': 9}, 'label': 'Old Double DQN', 'best_episode': 200, 'weights_unchanged': True, 'checkpoint_sha256': 'b616b5c8fdb10587a8b9054484348ee0db118b02cf2b4e6e5e54cba9d90d7379', 'per_group': {'rook': {'wins': 27, 'losses': 0, 'draws': 52, 'truncations': 1, 'games': 80, 'win_rate': 0.3375, 'reasons': {'repetition': 46, 'checkmate': 25, 'stalemate': 2, 'no_capture': 6, 'ply_limit': 1}}, 'cannon': {'wins': 6, 'losses': 0, 'draws': 48, 'truncations': 6, 'games': 60, 'win_rate': 0.1, 'reasons': {'no_capture': 10, 'repetition': 38, 'checkmate': 6, 'ply_limit': 6}}, 'horse': {'wins': 0, 'losses': 0, 'draws': 58, 'truncations': 2, 'games': 60, 'win_rate': 0.0, 'reasons': {'repetition': 54, 'no_capture': 4, 'ply_limit': 2}}}, 'per_board': {'rook-01': {'wins': 8, 'losses': 0, 'draws': 12, 'truncations': 0, 'games': 20, 'win_rate': 0.4, 'reasons': {'repetition': 12, 'checkmate': 7, 'stalemate': 1}}, 'rook-02': {'wins': 6, 'losses': 0, 'draws': 14, 'truncations': 0, 'games': 20, 'win_rate': 0.3, 'reasons': {'repetition': 12, 'checkmate': 5, 'no_capture': 2, 'stalemate': 1}}, 'rook-03': {'wins': 7, 'losses': 0, 'draws': 13, 'truncations': 0, 'games': 20, 'win_rate': 0.35, 'reasons': {'repetition': 11, 'checkmate': 7, 'no_capture': 2}}, 'rook-04': {'wins': 6, 'losses': 0, 'draws': 13, 'truncations': 1, 'games': 20, 'win_rate': 0.3, 'reasons': {'checkmate': 6, 'ply_limit': 1, 'repetition': 11, 'no_capture': 2}}, 'cannon-01': {'wins': 2, 'losses': 0, 'draws': 16, 'truncations': 2, 'games': 20, 'win_rate': 0.1, 'reasons': {'no_capture': 4, 'repetition': 12, 'checkmate': 2, 'ply_limit': 2}}, 'cannon-02': {'wins': 3, 'losses': 0, 'draws': 16, 'truncations': 1, 'games': 20, 'win_rate': 0.15, 'reasons': {'repetition': 13, 'checkmate': 3, 'no_capture': 3, 'ply_limit': 1}}, 'cannon-03': {'wins': 1, 'losses': 0, 'draws': 16, 'truncations': 3, 'games': 20, 'win_rate': 0.05, 'reasons': {'repetition': 13, 'no_capture': 3, 'ply_limit': 3, 'checkmate': 1}}, 'horse-01': {'wins': 0, 'losses': 0, 'draws': 19, 'truncations': 1, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 18, 'no_capture': 1, 'ply_limit': 1}}, 'horse-02': {'wins': 0, 'losses': 0, 'draws': 20, 'truncations': 0, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 18, 'no_capture': 2}}, 'horse-03': {'wins': 0, 'losses': 0, 'draws': 19, 'truncations': 1, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 18, 'no_capture': 1, 'ply_limit': 1}}}}, 'new_double_dqn': {'wins': 18, 'losses': 0, 'draws': 167, 'truncations': 15, 'games': 200, 'win_rate': 0.09, 'reasons': {'repetition': 153, 'checkmate': 16, 'stalemate': 2, 'no_capture': 14, 'ply_limit': 15}, 'label': 'New Double DQN (20 boards)', 'best_episode': 100, 'weights_unchanged': True, 'checkpoint_sha256': 'f99cdf676085cbdc8d367d315ff2b5a0c63a9b0aa2171292d1f07cd576887bd6', 'per_group': {'rook': {'wins': 18, 'losses': 0, 'draws': 61, 'truncations': 1, 'games': 80, 'win_rate': 0.225, 'reasons': {'repetition': 57, 'checkmate': 16, 'stalemate': 2, 'no_capture': 4, 'ply_limit': 1}}, 'cannon': {'wins': 0, 'losses': 0, 'draws': 50, 'truncations': 10, 'games': 60, 'win_rate': 0.0, 'reasons': {'repetition': 45, 'no_capture': 5, 'ply_limit': 10}}, 'horse': {'wins': 0, 'losses': 0, 'draws': 56, 'truncations': 4, 'games': 60, 'win_rate': 0.0, 'reasons': {'repetition': 51, 'no_capture': 5, 'ply_limit': 4}}}, 'per_board': {'rook-01': {'wins': 4, 'losses': 0, 'draws': 16, 'truncations': 0, 'games': 20, 'win_rate': 0.2, 'reasons': {'repetition': 16, 'checkmate': 3, 'stalemate': 1}}, 'rook-02': {'wins': 8, 'losses': 0, 'draws': 12, 'truncations': 0, 'games': 20, 'win_rate': 0.4, 'reasons': {'checkmate': 7, 'no_capture': 2, 'repetition': 10, 'stalemate': 1}}, 'rook-03': {'wins': 4, 'losses': 0, 'draws': 16, 'truncations': 0, 'games': 20, 'win_rate': 0.2, 'reasons': {'repetition': 16, 'checkmate': 4}}, 'rook-04': {'wins': 2, 'losses': 0, 'draws': 17, 'truncations': 1, 'games': 20, 'win_rate': 0.1, 'reasons': {'checkmate': 2, 'repetition': 15, 'ply_limit': 1, 'no_capture': 2}}, 'cannon-01': {'wins': 0, 'losses': 0, 'draws': 16, 'truncations': 4, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 14, 'no_capture': 2, 'ply_limit': 4}}, 'cannon-02': {'wins': 0, 'losses': 0, 'draws': 16, 'truncations': 4, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 15, 'ply_limit': 4, 'no_capture': 1}}, 'cannon-03': {'wins': 0, 'losses': 0, 'draws': 18, 'truncations': 2, 'games': 20, 'win_rate': 0.0, 'reasons': {'ply_limit': 2, 'repetition': 16, 'no_capture': 2}}, 'horse-01': {'wins': 0, 'losses': 0, 'draws': 19, 'truncations': 1, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 16, 'no_capture': 3, 'ply_limit': 1}}, 'horse-02': {'wins': 0, 'losses': 0, 'draws': 17, 'truncations': 3, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 15, 'ply_limit': 3, 'no_capture': 2}}, 'horse-03': {'wins': 0, 'losses': 0, 'draws': 20, 'truncations': 0, 'games': 20, 'win_rate': 0.0, 'reasons': {'repetition': 20}}}}}, 'torch_version': '2.14.1+cpu', 'python_version': '3.12.14 (main, Aug 25 2026, 14:01:42) [MSC v.1944 64 bit (AMD64)]', 'plan_sha256': 'eada2ccd11ad727118fe330f7881b14e4d8b2d5d61431ffcf4612d0fac00bf9f', 'board_bank_sha256': '21965057bc63b1df9d90e6d400d83d0eb2b75bc6f936782c347b5f09590ff045', 'seconds': 1303.7824080999708, 'paired_seed_check': True, 'old_results_reproduced_exactly': {'old_dqn': True, 'old_double_dqn': True}, 'comparisons': {'dqn': {'win_change': 2, 'win_rate_change_pp': 0.9999999999999996, 'new_win_old_not': 21, 'old_win_new_not': 19, 'per_group_change_pp': {'rook': -3.7500000000000004, 'cannon': 8.333333333333332, 'horse': 0.0}}, 'double_dqn': {'win_change': -15, 'win_rate_change_pp': -7.500000000000001, 'new_win_old_not': 12, 'old_win_new_not': 27, 'per_group_change_pp': {'rook': -11.250000000000002, 'cannon': -10.0, 'horse': 0.0}}}, 'scope': 'Previously inspected test bank reused for a regression comparison; not a fresh blind holdout. No test-based checkpoint selection.'}
    EVALUATION_SOURCE = '"""Evaluate saved best policies after both runs, without training or selection."""\nfrom collections import Counter\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport torch\nfrom dqn_model import QNetwork, encode_state, greedy_action\nfrom dqn_training import FixedOpponentEnv\n\n\ndef run_frozen_comparison(base_dir, runs, test_data, reference):\n    base_dir = Path(base_dir)\n    assert (base_dir / "completed.json").is_file(), "Finish both training runs first."\n    folder = base_dir / "test200"\n    folder.mkdir(exist_ok=True)\n    (folder / "episodes").mkdir(exist_ok=True)\n    cases = [(b["id"], seed) for b in test_data["boards"] for seed in b["opponent_seeds"]]\n    assert len(cases) == 200 and len(set(cases)) == 200\n    training_bank = json.loads((base_dir / "boards.json").read_text())\n    training_starts = {" ".join(b["fen"].split()[:2]) for b in training_bank["training"] + training_bank["validation"]}\n    assert not training_starts.intersection(" ".join(b["fen"].split()[:2]) for b in test_data["boards"])\n    report = {"status": "running", "training": False, "epsilon": 0, "games_per_model": 200,\n              "scope": "Previously inspected ten-board regression bank; no test-based checkpoint selection.",\n              "results": {}, "references": reference["results"]}\n    torch.set_num_threads(2)\n    with torch.inference_mode():\n        for algorithm, run_name in runs:\n            checkpoint = base_dir / run_name / "best.pt"\n            before_hash = hashlib.sha256(checkpoint.read_bytes()).hexdigest()\n            saved = torch.load(checkpoint, map_location="cpu", weights_only=True)\n            assert saved["config"]["algorithm"] == algorithm\n            assert saved["config"]["max_plies"] == 160 and saved["config"]["learner"] == "red"\n            model = QNetwork(**saved["network_config"])\n            model.load_state_dict(saved["model"])\n            model.eval().requires_grad_(False)\n            before = {k: v.clone() for k, v in model.state_dict().items()}\n            rows = []\n            for board in test_data["boards"]:\n                for repeat, seed in enumerate(board["opponent_seeds"], 1):\n                    env = FixedOpponentEnv(160, "red")\n                    obs, info = env.reset(board["fen"], seed)\n                    while info["outcome"] is None:\n                        action = greedy_action(model, encode_state(obs), obs["legal_action_ids"], "cpu")\n                        obs, _, _, _, info = env.step(action)\n                    outcome = info["outcome"]\n                    result = ("truncations" if outcome["truncated"] else "draws" if outcome["winner"] is None\n                              else "wins" if outcome["winner"] == "red" else "losses")\n                    rows.append(dict(model=run_name, board=board["id"], group=board["group"], repeat=repeat,\n                                     seed=seed, result=result, reason=outcome["reason"],\n                                     plies=len(env.game.history), final_fen=env.game.fen))\n                    episode = env.game.export()\n                    episode["evaluation"] = dict(model=run_name, checkpoint_sha256=before_hash,\n                                                 board=board["id"], seed=seed, training=False, epsilon=0)\n                    (folder / "episodes" / f"{run_name}_{board[\'id\']}_{repeat:02d}.json").write_text(json.dumps(episode))\n                print(f"TEST {run_name}: {len(rows)}/200 games", flush=True)\n            assert [(r["board"], r["seed"]) for r in rows] == cases\n            assert all(torch.equal(v, model.state_dict()[k]) for k, v in before.items())\n            assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == before_hash\n            with (folder / f"{run_name}_games.csv").open("w", newline="") as stream:\n                writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n                writer.writeheader()\n                writer.writerows(rows)\n            def stats(selected):\n                counts = Counter(r["result"] for r in selected)\n                return {**{k: counts[k] for k in ("wins", "losses", "draws", "truncations")},\n                        "games": len(selected), "win_rate": counts["wins"] / len(selected),\n                        "reasons": dict(Counter(r["reason"] for r in selected))}\n            result = {**stats(rows), "best_episode": saved["episode"], "weights_unchanged": True,\n                      "checkpoint_sha256": before_hash,\n                      "per_group": {g: stats([r for r in rows if r["group"] == g]) for g in ("rook", "cannon", "horse")}}\n            previous = reference["results"]["new_" + algorithm]\n            result["win_rate_change_pp_vs_uniform20"] = 100 * (result["win_rate"] - previous["win_rate"])\n            report["results"][algorithm] = result\n            (folder / "comparison.json").write_text(json.dumps(report, indent=2))\n            print("TEST RESULT", algorithm, json.dumps(result), flush=True)\n    report["status"] = "complete"\n    (folder / "comparison.json").write_text(json.dumps(report, indent=2))\n    (folder / "boards.json").write_text(json.dumps(test_data, indent=2))\n    package = base_dir / (base_dir.name + "_RESULTS.zip")\n    with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(folder.rglob("*")):\n            if path.is_file():\n                archive.write(path, path.relative_to(base_dir))\n        for _, run_name in runs:\n            for path in sorted((base_dir / run_name).glob("*")):\n                if path.suffix in (".json", ".csv", ".zip"):\n                    archive.write(path, path.relative_to(base_dir))\n        for name in ("completed.json", "boards.json", "source_hashes.json"):\n            archive.write(base_dir / name, name)\n    return package, report\n'
    exec(compile(EVALUATION_SOURCE, "terminal_replay_evaluation.py", "exec"))
    comparison_zip, comparison = run_frozen_comparison(BASE_DIR, RUNS, TEST_DATA, REFERENCE)
    data = comparison_zip.read_bytes()
    digest = hashlib.sha256(data).hexdigest()
    encoded = base64.b64encode(data).decode("ascii")
    display(HTML(f'<p>Both runs and 400 frozen test games complete.</p><a download="{comparison_zip.name}" data-sha256="{digest}" href="data:application/zip;base64,{encoded}">Save models and test results</a>'))
    print("TRAINING AND FROZEN TESTS COMPLETE", flush=True)
